1. Import Libraries

In [1]:
## kernel - ids
"""
Persist Salesforce object field metadata (df_meta) into a SQL Server table.

- Creates the table if it doesn't exist
- Appends new rows OR performs an UPSERT (MERGE) if enabled
- Uses fast_executemany for performance
"""

#Importing Other Libraries
from simple_salesforce import Salesforce,SalesforceLogin,SalesforceAuthenticationFailed,SFType
import pandas as pd
import json
import numpy as np
import time as t
import uuid
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from collections import OrderedDict

from core_functions import *
from sf_source_objects import *
from sf_target_objects import *

In [3]:
# ----------------------------
# PARAMETERS
# 1) CONFIGURATION
# ----------------------------
p_db_server  = 'Rhombus-SQL-Ser'
p_db  = 'SalesforceMigration'
p_sch = 'ids'
p_tbl_fd = 'sf_object_fd'
p_tbl_pl = 'sf_object_pl'
p_sf_src = 'hubprod'
# p_sf_src = 'hubuat'
# p_sf_tgt = 'rhpartial'
# p_sf_tgt = 'rhdata'
# p_sf_tgt = 'rhuat'
# p_sf_tgt = 'datasb'
# p_sf_tgt = 'flow2sb'
p_sf_tgt = 'rhsit'

# ----------------------------
# CONSTANTS
# WRITE: APPEND or UPSERT
# ----------------------------
UPSERT_ENABLED = True  # set False to just append/replace
WRITE_CHUNK_SIZE = 10_000

connection_string = (
    f"mssql+pyodbc://@{p_db_server}/{p_db}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&fast_executemany=True"
)


## Configure Logging
This section initializes a logging framework to monitor the ETL workflow’s execution and facilitate debugging by:
Creating an isolated logger named ETL_Salesforce.
Setting the log level to INFO to capture all important messages.
Outputting formatted messages (with timestamps and levels) to the console.
Disabling propagation to prevent duplicate log entrie

In [5]:
# Configuring logging
import logging
 
# Create a dedicated logger for the ETL process
logger = logging.getLogger('ETL_Salesforce')
logger.setLevel(logging.INFO)
 
# Clear any existing handlers to avoid duplicates
logger.handlers = []  # Remove all handlers from this logger
 
# Create a new StreamHandler with the desired format
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(handler)
 
# Disable propagation to prevent root logger interference
logger.propagate = False

## Establish Connections
This section establishes a connection to the source Salesforce environment using credentials securely fetched from Google Colab’s secrets storage.
Connecting to the Source Salesforce Instance:

In [4]:
#Connecting To The Source Salesforce Instance
from secrets_ids import secrets
url = secrets.get(p_sf_src, {}).get('site', 'Unknown')
username = secrets.get(p_sf_src, {}).get('username', 'Unknown')
pswd = secrets.get(p_sf_src, {}).get('password', 'Unknown')
token = secrets.get(p_sf_src, {}).get('token', 'Unknown')

src_sf_instance = connect_to_salesforce(url=url, username=username, password=pswd, token=token, domain='login', logger=logger)
 
logger.info(f"Source SF Instance Session ID = {src_sf_instance.session_id}")

2026-09-03 11:41:45,660 [INFO] Connected successfully to https://ioofadviserhub.my.salesforce.com
2026-09-03 11:41:45,661 [INFO] Source SF Instance Session ID = 00D6F0000029Bzd!AQEAQFvKAVv8xUIEq9pP4LuifIq4ybyy9LLCUzK094I_fMS87.vmiV.MtJ2rIWSurGeD1.x76u8DOoiIcEyzBOMd0rYOUIzs


To maintain security, credentials are not hardcoded. Instead, they are stored in Google Colab’s Secrets vault. The application dynamically accesses these stored values (src_username, src_password, and src_token) via the userdata.get('secret_name') method. The fetched credentials are passed to the connect_to_salesforce function to authenticate and create a Salesforce client instance.

Connecting to the Destination Salesforce Instance:

In [6]:
#Connecting To The Destination Salesforce Instance
from secrets_ids import secrets
url_tgt = secrets.get(p_sf_tgt, {}).get('site', 'Unknown')
username_tgt = secrets.get(p_sf_tgt, {}).get('username', 'Unknown')
pswd_tgt = secrets.get(p_sf_tgt, {}).get('password', 'Unknown')
token_tgt = secrets.get(p_sf_tgt, {}).get('token', 'Unknown')

dst_sf_instance = connect_to_salesforce(url=url_tgt, username=username_tgt, password=pswd_tgt, token=token_tgt, domain='test', logger=logger)
 
logger.info(f"Destination SF Instance Session ID = {dst_sf_instance.session_id}")

2026-09-04 00:12:38,772 [INFO] Connected successfully to https://test.salesforce.com
2026-09-04 00:12:38,773 [INFO] Destination SF Instance Session ID = 00DBm00000CWPlx!AQEAQJisfayl5n7.FwzSKP.ctprm4b84oZySLRyBaIa8MAzE8jvHcOxYPPc05qLr3sPwjfiGuz76lF5JCcIqb2.XZT8KKTbS


## Phase 2 - User - Internal, not associated with Account and Contact
### Only update recods with matched Metazoa_External_Id__c

In [12]:
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_User-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields = USER_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total User Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='User', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000,logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='User', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000,logger=logger,mode='UPSERT')
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='User', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000,logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:17:49,666 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:17:49,666 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:17:49,666 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:17:49,666 INFO sqlalchemy.engine.Engine [generated in 0.00159s] ()
2026-09-03 17:17:49,674 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:17:49,675 INFO sqlalchemy.engine.Engine [generated in 0.00100s] ()
2026-09-03 17:17:49,677 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:17:49,678 INFO sqlalchemy.engine.Engine [generated in 0.00079s] ()
2026-09-03 17:17:49,680 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:17:49,681 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_User-ALL]
;
2026-09-03 17:17:49,682 INFO sqlalchemy.engine.Engine 

2026-09-03 17:17:50,112 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 17:17:50,116 [INFO] Total User Records: 33
2026-09-03 17:17:50,117 [INFO] Unique in Metazoa_External_ID__c: 33
2026-09-03 17:17:50,133 [INFO] Preparing to upsert 33 records to User with fields ['Metazoa_External_ID__c', 'AboutMe', 'Alias', 'City', 'ContactId', 'Country', 'DefaultGroupNotificationFrequency', 'DelegatedApproverId', 'Department', 'DigestFrequency', 'Division', 'DNU_Referral_Partner_Visibility__c', 'EDP_External_Id__c', 'Email', 'Email_Signature__c', 'EmailEncodingKey', 'EmailPreferencesAutoBcc', 'EmailPreferencesAutoBccStayInTouch', 'EmailPreferencesStayInTouchReminder', 'EmployeeNumber', 'EndDay', 'Extension', 'Fax', 'FederationIdentifier', 'FirstName', 'ForecastEnabled', 'GeocodeAccuracy', 'Internal_ReferralPartner__c', 'IsActive', 'LanguageLocaleKey', 'LastName', 'Latitude', 'Licensee__c', 'licenseesupervisor__c', 'LocaleSidKey', 'Location__c', 'Longitude', 'Manager__c', 'ManagerId', 'MobilePhone', 'Phone', 'PortalRole', 'Position__c', 'PostalCode', 'Receive

## OBSOLETE - DON'T RUN - PermissionSet Object
**PermissionSet will be handled outside of this script.**

In [ ]:
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_PermissionSet-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields = PERMISSIONSET_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total PermissionSet Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='PermissionSet', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Account Object

In [14]:
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Account-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields = ACCOUNT_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Account Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Account', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:25:57,022 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:25:57,023 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:25:57,025 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:25:57,026 INFO sqlalchemy.engine.Engine [generated in 0.00081s] ()
2026-09-03 17:25:57,028 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:25:57,029 INFO sqlalchemy.engine.Engine [generated in 0.00079s] ()
2026-09-03 17:25:57,031 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:25:57,032 INFO sqlalchemy.engine.Engine [generated in 0.00087s] ()
2026-09-03 17:25:57,034 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:25:57,035 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Account-ALL]
;
2026-09-03 17:25:57,035 INFO sqlalchemy.engine.Engi

2026-09-03 17:25:57,238 [INFO] Total Account Records: 141
2026-09-03 17:25:57,245 [INFO] Unique in Metazoa_External_ID__c: 141
2026-09-03 17:25:57,272 [INFO] Preparing to upsert 141 records to Account with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedDate', 'CreatedById', 'LastModifiedDate', 'LastModifiedById', 'Name', 'Type', 'RecordTypeId', 'BillingStreet', 'BillingCity', 'BillingState', 'BillingPostalCode', 'BillingCountry', 'BillingLatitude', 'BillingLongitude', 'BillingGeocodeAccuracy', 'ShippingStreet', 'ShippingCity', 'ShippingState', 'ShippingPostalCode', 'ShippingCountry', 'ShippingLatitude', 'ShippingLongitude', 'ShippingGeocodeAccuracy', 'Phone', 'Fax', 'AccountNumber', 'Website', 'Sic', 'AnnualRevenue', 'NumberOfEmployees', 'Ownership', 'TickerSymbol', 'Description', 'Rating', 'Site', 'SourceSystemIdentifier', 'Jigsaw', 'Industry', 'AccountSource', 'SicDesc', 'ABN__c', 'ACN__c', 'AFSL_of_Previous_Licensee__c', 'ANZ_Lookback_No_of_Fail_Outcomes__c', 'ANZ_Lookback_No_

## Phase 2 - Contact - Round 1 - Base Contact records placeholder for Portal_User_Id__c

In [15]:
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Contact-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields = CONTACT_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Contact Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Contact', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Contact', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger,mode='UPSERT')
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Contact', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:27:41,153 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:27:41,154 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:27:41,156 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:27:41,157 INFO sqlalchemy.engine.Engine [generated in 0.00086s] ()
2026-09-03 17:27:41,161 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:27:41,161 INFO sqlalchemy.engine.Engine [generated in 0.00061s] ()
2026-09-03 17:27:41,163 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:27:41,164 INFO sqlalchemy.engine.Engine [generated in 0.00058s] ()
2026-09-03 17:27:41,166 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:27:41,168 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Contact-ALL]
;
2026-09-03 17:27:41,170 INFO sqlalchemy.engine.Engi

2026-09-03 17:27:42,205 [INFO] Total Contact Records: 1380
2026-09-03 17:27:42,207 [INFO] Unique in Metazoa_External_ID__c: 1380
2026-09-03 17:27:42,488 [INFO] Preparing to upsert 1380 records to Contact with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedDate', 'CreatedById', 'LastModifiedDate', 'LastModifiedById', 'Adviser_Client_Release_Required__c', 'Adviser_Recognition_Program__c', 'Advisory_Board_Member__c', 'Departed__c', 'DoNotCall', 'HasOptedOutOfEmail', 'HasOptedOutOfFax', 'Wholesale_Client_Adviser__c', 'X2022_Education_Exemption__c', 'Recovery_Charge_Rate__c', 'AFP_Check_Date__c', 'ANZ_Employment_Start_Date__c', 'ASIC_Ceased_Date__c', 'ASIC_Start_Date__c', 'ASN_Start_Date__c', 'Birthdate', 'Date_when_Attestation_was_Received__c', 'Expected_Leave_Start_Date__c', 'Expected_Return_Date__c', 'Induction_Complete__c', 'Industry_Exam_Pass_Date__c', 'Last_AQR_Date__c', 'Last_Assurance_Review_Date__c', 'Last_Debrief_Date__c', 'Last_General_Advice_Review_Date__c', 'Onboarded_On_

## Phase 2 - User - Community, associated with Account and Contact

In [16]:
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_User-Community]
;"""

from sf_target_fields import *

ls_tgt_objFields= USER_TARGET_FIELDS
ls_tgt_objFields = [x for x in ls_tgt_objFields if x not in ('UserRoleId')]

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total User Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='User', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:29:37,164 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:29:37,165 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:29:37,167 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:29:37,168 INFO sqlalchemy.engine.Engine [generated in 0.00074s] ()
2026-09-03 17:29:37,171 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:29:37,172 INFO sqlalchemy.engine.Engine [generated in 0.00110s] ()
2026-09-03 17:29:37,174 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:29:37,175 INFO sqlalchemy.engine.Engine [generated in 0.00109s] ()
2026-09-03 17:29:37,178 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:29:37,180 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_User-Community]
;
2026-09-03 17:29:37,181 INFO sqlalchemy.engine.E

2026-09-03 17:29:48,198 [INFO] Total User Records: 822
2026-09-03 17:29:48,200 [INFO] Unique in Metazoa_External_ID__c: 822
2026-09-03 17:29:48,282 [INFO] Preparing to upsert 822 records to User with fields ['Metazoa_External_ID__c', 'AboutMe', 'Alias', 'City', 'ContactId', 'Country', 'DefaultGroupNotificationFrequency', 'DelegatedApproverId', 'Department', 'DigestFrequency', 'Division', 'DNU_Referral_Partner_Visibility__c', 'EDP_External_Id__c', 'Email', 'Email_Signature__c', 'EmailEncodingKey', 'EmailPreferencesAutoBcc', 'EmailPreferencesAutoBccStayInTouch', 'EmailPreferencesStayInTouchReminder', 'EmployeeNumber', 'EndDay', 'Extension', 'Fax', 'FederationIdentifier', 'FirstName', 'ForecastEnabled', 'GeocodeAccuracy', 'Internal_ReferralPartner__c', 'IsActive', 'LanguageLocaleKey', 'LastName', 'Latitude', 'Licensee__c', 'licenseesupervisor__c', 'LocaleSidKey', 'Location__c', 'Longitude', 'Manager__c', 'ManagerId', 'MobilePhone', 'Phone', 'PortalRole', 'Position__c', 'PostalCode', 'Rece

## Phase 2 - User - Community, NOT associated with Account and Contact, INACTIVE, with user license 'Customer Community Plus' (Advisers) and 'Customer Community Plus Login' (Support Staff)
These users will be migrated to Rhombus as inactive users attached to a custom profile which does not require license.

In [17]:
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_User-InactiveCommunity]
;"""

from sf_target_fields import *

ls_tgt_objFields= USER_TARGET_FIELDS
ls_tgt_objFields = [x for x in ls_tgt_objFields if x not in ('UserRoleId')]

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total User Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='User', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=1000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:40:24,756 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:40:24,758 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:40:24,760 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:40:24,761 INFO sqlalchemy.engine.Engine [generated in 0.00097s] ()
2026-09-03 17:40:24,764 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:40:24,765 INFO sqlalchemy.engine.Engine [generated in 0.00100s] ()
2026-09-03 17:40:24,767 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:40:24,767 INFO sqlalchemy.engine.Engine [generated in 0.00082s] ()
2026-09-03 17:40:24,769 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:40:24,770 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_User-InactiveCommunity]
;
2026-09-03 17:40:24,771 INFO sqlalchemy.

2026-09-03 17:40:25,058 [INFO] Total User Records: 4
2026-09-03 17:40:25,059 [INFO] Unique in Metazoa_External_ID__c: 4
2026-09-03 17:40:25,071 [INFO] Preparing to upsert 4 records to User with fields ['Metazoa_External_ID__c', 'AboutMe', 'Alias', 'City', 'ContactId', 'Country', 'DefaultGroupNotificationFrequency', 'DelegatedApproverId', 'Department', 'DigestFrequency', 'Division', 'DNU_Referral_Partner_Visibility__c', 'EDP_External_Id__c', 'Email', 'Email_Signature__c', 'EmailEncodingKey', 'EmailPreferencesAutoBcc', 'EmailPreferencesAutoBccStayInTouch', 'EmailPreferencesStayInTouchReminder', 'EmployeeNumber', 'EndDay', 'Extension', 'Fax', 'FederationIdentifier', 'FirstName', 'ForecastEnabled', 'GeocodeAccuracy', 'Internal_ReferralPartner__c', 'IsActive', 'LanguageLocaleKey', 'LastName', 'Latitude', 'Licensee__c', 'licenseesupervisor__c', 'LocaleSidKey', 'Location__c', 'Longitude', 'Manager__c', 'ManagerId', 'MobilePhone', 'Phone', 'PortalRole', 'Position__c', 'PostalCode', 'ReceivesAd

## Phase 2 - PermissionSetAssignment Object (for Community Users)

In [18]:
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_PermissionSetAssignment-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields = PERMISSIONSETASSIGNMENT_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total PermissionSetAssignment Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='PermissionSetAssignment', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:45:48,497 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:45:48,498 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:45:48,501 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:45:48,502 INFO sqlalchemy.engine.Engine [generated in 0.00117s] ()
2026-09-03 17:45:48,505 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:45:48,506 INFO sqlalchemy.engine.Engine [generated in 0.00083s] ()


2026-09-03 17:45:48,508 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:45:48,508 INFO sqlalchemy.engine.Engine [generated in 0.00087s] ()
2026-09-03 17:45:48,510 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:45:48,511 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_PermissionSetAssignment-ALL]
;
2026-09-03 17:45:48,512 INFO sqlalchemy.engine.Engine [generated in 0.00109s] ()
2026-09-03 17:45:49,091 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 17:45:49,094 [INFO] Total PermissionSetAssignment Records: 645
2026-09-03 17:45:49,098 [INFO] Preparing to upsert 645 records to PermissionSetAssignment with fields ['ExpirationDate', 'PermissionSetId', 'PermissionSetGroupId', 'AssigneeId']
2026-09-03 17:46:22,762 [INFO] Upsert complete. Time taken: 33.66s
2026-09-03 17:46:22,763 [INFO] Success: 645, Failures: 0
2026-09-03 17:46:22,770 [INFO] Load Results:      success  created                  id errors
0       True     True  0PaBm00000RA5OfKAL     []
1       True     True  0PaBm00000RA5OgKAL     []
2       True     True  0PaBm00000RA5OhKAL     []
3       True     True  0PaBm00000RA5OiKAL     []
4       True     True  0PaBm00000RA5OjKAL     []
..       ...      ...                 ...    ...
640     True     True  0PaBm00000RA5dpKAD     []
641     True     True  0PaBm00000RA5dqKAD     []
642     True     True  0PaBm00000RA5drKAD     []
643     True     True  0PaBm00000RA5dsKAD     []
644     True     True  0PaBm00000RA5dtKA

## OBSOLETE - User - Licensee Supervisors and Consultants - UPSERT to correct Email
These users have been migrated to Rhombus in 2025. Their User record may still have invalid email address which can be updated from current IFL

In [6]:
# object transfer from source to target.
q_sql_select_src_tbl = f"""with cte_email as (
SELECT 
	[Id-tgt] = t.[Id]
	,[Email-src] = s.[Email]
	,[Email-tgt] = t.[Email]
FROM [SalesforceMigration].[sf].[User] s
join [SalesforceMigration].[sfrhd].[User] t
	on t.[Metazoa_External_ID__c] = s.[Id]
)
select top 1 [Id] = e.[Id-tgt], e.[Email-src], [Email] = e.[Email-src]+'.invalid'
from [sfrhd].[Account] a
join [cte_email] e on e.[Id-tgt] = a.[Licensee_Supervisor__c]
where 1=1
and e.[Email-src] not like 'venkata.vemireddy%'
and e.[Email-tgt] like '0%invalid'
;"""

ls_tgt_objFields = ['Id', 'Email']

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

2026-09-02 17:25:21,655 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-02 17:25:21,656 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-02 17:25:21,658 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-02 17:25:21,659 INFO sqlalchemy.engine.Engine [generated in 0.00110s] ()


2026-09-02 17:25:21,662 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-02 17:25:21,663 INFO sqlalchemy.engine.Engine [generated in 0.00082s] ()
2026-09-02 17:25:21,665 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-02 17:25:21,666 INFO sqlalchemy.engine.Engine [generated in 0.00067s] ()
2026-09-02 17:25:21,669 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-02 17:25:21,671 INFO sqlalchemy.engine.Engine with cte_email as (
SELECT 
	[Id-tgt] = t.[Id]
	,[Email-src] = s.[Email]
	,[Email-tgt] = t.[Email]
FROM [SalesforceMigration].[sf].[User] s
join [SalesforceMigration].[sfrhd].[User] t
	on t.[Metazoa_External_ID__c] = s.[Id]
)
select top 1 [Id] = e.[Id-tgt], e.[Email-src], [Email] = e.[Email-src]+'.invalid'
from [sfrhd].[Account] a
join [cte_email] e on e.[Id-tgt] = a.[Licensee_Supervisor__c]
where 1=1
and e.[Email-src] not like 'venkata.vemireddy%'
an

,Id,Email
0,005W2000000K2eHIAS,peter.purpura@ioof.com.au.invalid


In [7]:

logger.info(f"Total User Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='User', reference_field='Id', fields=ls_tgt_objFields, batch_size=1000, logger=logger,mode='UPSERT')
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-02 17:25:56,129 [INFO] Total User Records: 1
2026-09-02 17:25:56,134 [INFO] Preparing to upsert 1 records to User with fields ['Id', 'Email']
2026-09-02 17:26:03,241 [INFO] Upsert complete. Time taken: 7.11s
2026-09-02 17:26:03,242 [INFO] Success: 1, Failures: 0
2026-09-02 17:26:03,246 [INFO] Load Results:    success  created                  id errors
0     True    False  005W2000000K2eHIAS     []


## Phase 2 - Contact - Round 2 - Populate Portal_User_Id__c using UPSERT

In [19]:
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Contact-UPSERT] where [Portal_User_ID__c] is not null
;"""

ls_tgt_objFields = [
'Id'
,'Portal_User_ID__c'
,'Industry_Exam_Status__c'
]


engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Contact Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Contact', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Contact', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger,mode='UPSERT')
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Contact', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:47:31,940 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:47:31,949 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:47:31,951 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:47:31,952 INFO sqlalchemy.engine.Engine [generated in 0.00110s] ()
2026-09-03 17:47:31,955 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:47:31,956 INFO sqlalchemy.engine.Engine [generated in 0.00081s] ()
2026-09-03 17:47:31,958 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:47:31,958 INFO sqlalchemy.engine.Engine [generated in 0.00072s] ()
2026-09-03 17:47:31,960 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026-09-03 17:47:31,962 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Contact-UPSERT] where [Portal_User_ID__c] is not null
;
2026-09-03 17:47:31,962 INFO sqlalchemy.engine.Engine [generated in 0.00104s] ()
2026-09-03 17:47:32,751 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 17:47:32,754 [INFO] Total Contact Records: 821
2026-09-03 17:47:32,760 [INFO] Preparing to upsert 821 records to Contact with fields ['Id', 'Portal_User_ID__c', 'Industry_Exam_Status__c']
2026-09-03 17:47:44,913 [INFO] Upsert complete. Time taken: 12.15s
2026-09-03 17:47:44,915 [INFO] Success: 821, Failures: 0
2026-09-03 17:47:44,920 [INFO] Load Results:      success  created                  id errors
0       True    False  003Bm00001HKuu8IAD     []
1       True    False  003Bm00001HKuuNIAT     []
2       True    False  003Bm00001HKuuLIAT     []
3       True    False  003Bm00001HKuqMIAT     []
4       True    False  003Bm00001HKuqjIAD     []
..       ...      ...                 ...    ...
816     True    False  003Bm00001HKukBIAT     []
817     True    False  003Bm00001HKukiIAD     []
818     True    False  003Bm00001HKul5IAD     []
819     True    False  003Bm00001HKulAIAT     []
820     True    False  003Bm00001HKuk0IAD     []

[821 rows x 4 columns]


## 20 After User - AccountContactRelation

In [20]:
"""
reference_to
["Account"]
["Contact"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_AccountContactRelation-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= AccountContactRelation_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total AccountContactRelation Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:48:54,471 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:48:54,472 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:48:54,474 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:48:54,475 INFO sqlalchemy.engine.Engine [generated in 0.00111s] ()
2026-09-03 17:48:54,477 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:48:54,478 INFO sqlalchemy.engine.Engine [generated in 0.00101s] ()


2026-09-03 17:48:54,480 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:48:54,482 INFO sqlalchemy.engine.Engine [generated in 0.00081s] ()
2026-09-03 17:48:54,482 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:48:54,482 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_AccountContactRelation-ALL]
;
2026-09-03 17:48:54,482 INFO sqlalchemy.engine.Engine [generated in 0.00174s] ()
2026-09-03 17:48:56,439 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 17:48:56,439 [INFO] Total AccountContactRelation Records: 1538
2026-09-03 17:48:56,439 [INFO] Unique in Metazoa_External_ID__c: 1538
2026-09-03 17:48:56,473 [INFO] Preparing to upsert 1538 records to AccountContactRelation with fields ['Metazoa_External_ID__c', 'Comments__c', 'EDP_External_Id__c', 'Email__c', 'EndDate', 'IsActive', 'Relationship_Strength__c', 'Roles', 'StartDate', 'Status__c', 'AccountId', 'ContactId']
2026-09-03 17:49:08,301 [INFO] Upsert complete. Time taken: 11.83s
2026-09-03 17:49:08,303 [INFO] Success: 166, Failures: 1372
2026-09-03 17:49:08,309 [WARNING] Failures: [{'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'DUPLICATE_VALUE', 'message': 'This adviser already has a relationship with this practice.', 'fields': ['ContactId']}]}, {'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'DUPLICATE_VALUE', 'message': 'This adviser already has a relationship with this practice.', 'fields': ['ContactId']}]}, {'

## 20 After User - Third_Party_Provider__c

In [21]:
"""
reference_to
["Group", "User"]
["User"]
"""
"""
object_api_name
CommPay_Supplier__c
Email_Log__c
EventRelation
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Third_Party_Provider-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= Third_Party_Provider_TARGET_FIELDS
ls_tgt_objFields = [x for x in ls_tgt_objFields if x not in ('Name')]

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Third_Party_Provider__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Email_Address_1__c: {df_transformed['Email_Address_1__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Email_Log__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Third_Party_Provider__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=1000, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:50:30,454 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:50:30,455 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:50:30,457 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:50:30,458 INFO sqlalchemy.engine.Engine [generated in 0.00082s] ()


2026-09-03 17:50:30,460 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:50:30,462 INFO sqlalchemy.engine.Engine [generated in 0.00159s] ()
2026-09-03 17:50:30,464 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:50:30,465 INFO sqlalchemy.engine.Engine [generated in 0.00084s] ()
2026-09-03 17:50:30,467 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:50:30,475 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Third_Party_Provider-ALL]
;
2026-09-03 17:50:30,476 INFO sqlalchemy.engine.Engine [generated in 0.00119s] ()
2026-09-03 17:50:30,540 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 17:50:30,544 [INFO] Total Third_Party_Provider__c Records: 31
2026-09-03 17:50:30,547 [INFO] Preparing to upsert 31 records to Third_Party_Provider__c with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Email_Address_1__c', 'Email_Address_2__c', 'Email_Address_3__c', 'Third_Party_Provider_End_Date__c', 'Third_Party_Provider_Name__c', 'Third_Party_Provider_Start_Date__c']
2026-09-03 17:50:36,363 [INFO] Upsert complete. Time taken: 5.81s
2026-09-03 17:50:36,364 [INFO] Success: 31, Failures: 0
2026-09-03 17:50:36,372 [INFO] Load Results:     success  created                  id errors
0      True     True  a1lBm000002ke9NIAQ     []
1      True     True  a1lBm000002ke9OIAQ     []
2      True     True  a1lBm000002ke9PIAQ     []
3      True     True  a1lBm000002ke9QIAQ     []
4      True     True  a1lBm000002ke9RIAQ     []
5      True     True  a1lBm000002ke9SIAQ     []
6      True     True  a1lBm000002ke9TIAQ     []
7      True     True 

## 20 After User - SecondaryPracticeAdviserHistory__c

In [22]:
"""
reference_to
["Account"]
["Contact"]
["Group", "User"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_SecondaryPracticeAdviserHistory-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= SecondaryPracticeAdviserHistory_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total SecondaryPracticeAdviserHistory__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='SecondaryPracticeAdviserHistory__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:50:46,355 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:50:46,356 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:50:46,359 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:50:46,360 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()


2026-09-03 17:50:46,363 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:50:46,364 INFO sqlalchemy.engine.Engine [generated in 0.00082s] ()
2026-09-03 17:50:46,365 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:50:46,366 INFO sqlalchemy.engine.Engine [generated in 0.00077s] ()
2026-09-03 17:50:46,368 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:50:46,369 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_SecondaryPracticeAdviserHistory-ALL]
;
2026-09-03 17:50:46,370 INFO sqlalchemy.engine.Engine [generated in 0.00083s] ()
2026-09-03 17:50:46,490 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 17:50:46,494 [INFO] Total SecondaryPracticeAdviserHistory__c Records: 257
2026-09-03 17:50:46,495 [INFO] Unique in Metazoa_External_ID__c : 257
2026-09-03 17:50:46,503 [INFO] Preparing to upsert 257 records to SecondaryPracticeAdviserHistory__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'EDP_External_Id__c', 'End_Date__c', 'Roles__c', 'Start_Date__c', 'Practice__c', 'Adviser__c']
2026-09-03 17:50:52,420 [INFO] Upsert complete. Time taken: 5.92s
2026-09-03 17:50:52,421 [INFO] Success: 257, Failures: 0
2026-09-03 17:50:52,426 [INFO] Load Results:      success  created                  id errors
0       True     True  a1gBm000003cVQrIAM     []
1       True     True  a1gBm000003cVQsIAM     []
2       True     True  a1gBm000003cVQtIAM     []
3       True     True  a1gBm000003cVQuIAM     []
4       True     True  a1gBm000003cVQvIAM     []
..       ...      ...                 ...    ...
252     True     T

## 20 After User - Adviser_Academy__c

In [23]:
"""
reference_to
["Account"]
["Contact"]
["Contact"]
["RecordType"]
["User"]
"""
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Adviser_Academy-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= ADVISER_ACADEMY_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Adviser_Academy__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Adviser_Academy__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=1000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:52:20,154 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:52:20,156 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:52:20,157 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:52:20,157 INFO sqlalchemy.engine.Engine [generated in 0.00090s] ()
2026-09-03 17:52:20,162 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:52:20,162 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ()


2026-09-03 17:52:20,165 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:52:20,166 INFO sqlalchemy.engine.Engine [generated in 0.00151s] ()
2026-09-03 17:52:20,168 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:52:20,169 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Adviser_Academy-ALL]
;
2026-09-03 17:52:20,170 INFO sqlalchemy.engine.Engine [generated in 0.00103s] ()
2026-09-03 17:52:20,276 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 17:52:20,279 [INFO] Total Adviser_Academy__c Records: 4
2026-09-03 17:52:20,280 [INFO] Unique in Metazoa_External_ID__c: 4
2026-09-03 17:52:20,283 [INFO] Preparing to upsert 4 records to Adviser_Academy__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Practice__c', 'Principe_contact__c', 'Secondary_Contact__c', 'RecordTypeId', 'Advice_Coach__c', 'Advice_Consultant__c']
2026-09-03 17:52:26,243 [INFO] Upsert complete. Time taken: 5.96s
2026-09-03 17:52:26,243 [INFO] Success: 4, Failures: 0
2026-09-03 17:52:26,248 [INFO] Load Results:    success  created                  id errors
0     True     True  a0fBm000008fTpRIAU     []
1     True     True  a0fBm000008fTpSIAU     []
2     True     True  a0fBm000008fTpTIAU     []
3     True     True  a0fBm000008fTpUIAU     []


## 20 After User - CSO__c

In [24]:
"""
reference_to
["Group", "User"]
"""
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CSO-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= CSO_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total CSO__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CSO__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:53:02,711 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:53:02,712 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:53:02,714 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:53:02,715 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()
2026-09-03 17:53:02,718 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))


2026-09-03 17:53:02,718 INFO sqlalchemy.engine.Engine [generated in 0.00068s] ()
2026-09-03 17:53:02,720 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:53:02,723 INFO sqlalchemy.engine.Engine [generated in 0.00211s] ()
2026-09-03 17:53:02,723 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:53:02,723 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_CSO-ALL]
;
2026-09-03 17:53:02,723 INFO sqlalchemy.engine.Engine [generated in 0.00140s] ()
2026-09-03 17:53:02,799 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 17:53:02,803 [INFO] Total CSO__c Records: 151
2026-09-03 17:53:02,804 [INFO] Unique in Metazoa_External_ID__c: 151
2026-09-03 17:53:02,809 [INFO] Preparing to upsert 151 records to CSO__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Activity_Achieved__c', 'Activity_Exceed__c', 'Activity_Partial_achieved__c', 'Area__c', 'Business_City__c', 'Business_State_Province__c', 'Business_Street__c', 'Business_Zip_Postal_Code__c', 'CSOStatus__c', 'Email__c', 'First_Name__c', 'FTE__c', 'Last_Name__c', 'Mobile__c', 'Preferred_Name__c', 'Recovery_Charge_Rate__c', 'Responsibility__c', 'Salutation__c', 'Team_Leader__c', 'XPLAN_ID__c']
2026-09-03 17:53:16,832 [INFO] Upsert complete. Time taken: 14.02s
2026-09-03 17:53:16,833 [INFO] Success: 88, Failures: 63
2026-09-03 17:53:16,833 [WARNING] Failures: [{'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'INVALID_OR_NULL_FOR_RESTRICTED_PICKLIST',

## 20 After User - PSA__c

In [25]:
"""
reference_to
["Account"]
["Contact"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_PSA-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= PSA_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total PSA__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='PSA__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:54:10,453 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:54:10,454 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:54:10,457 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:54:10,457 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()
2026-09-03 17:54:10,461 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:54:10,462 INFO sqlalchemy.engine.Engine [generated in 0.00105s] ()
2026-09-03 17:54:10,464 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:54:10,464 INFO sqlalchemy.engine.Engine [generated in 0.00099s] ()
2026-09-03 17:54:10,464 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:54:10,464 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_PSA-ALL]
;
2026-09-03 17:54:10,464 INFO sqlalchemy.engine.Engine [

2026-09-03 17:54:10,561 [INFO] Total PSA__c Records: 0
2026-09-03 17:54:10,563 [INFO] Unique in Metazoa_External_ID__c: 0
2026-09-03 17:54:10,566 [INFO] Preparing to upsert 0 records to PSA__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Average_AQR_Score__c', 'Comments__c', 'EDP_External_Id__c', 'Is_Active__c', 'Number_Of_Open_Tasks__c', 'On_Watch_List_Manually__c', 'Review_Date__c', 'Review_Score__c', 'Status__c', 'Unanswered_Questions__c', 'Upcoming_Task_Due_Date__c', 'Practice_Name__c', 'Secondary_Contact__c', 'Licensee_Supervisor_Name__c']
2026-09-03 17:54:10,567 [ERROR] Error during upsert: data should not be empty for insert


ValueError: data should not be empty for insert

## 20 After User - Special_Conditions__c

In [26]:
"""
reference_to
["Contact"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Special_Conditions-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= SPECIAL_CONDITIONS_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Special_Conditions__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Special_Conditions__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:54:50,023 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:54:50,024 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:54:50,026 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:54:50,027 INFO sqlalchemy.engine.Engine [generated in 0.00104s] ()
2026-09-03 17:54:50,030 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:54:50,031 INFO sqlalchemy.engine.Engine [generated in 0.00138s] ()
2026-09-03 17:54:50,034 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:54:50,035 INFO sqlalchemy.engine.Engine [generated in 0.00142s] ()
2026-09-03 17:54:50,037 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:54:50,039 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Special_Conditions-ALL]
;
2026-09-03 17:54:50,040 INFO sqlalchemy.

2026-09-03 17:54:50,143 [INFO] Total Special_Conditions__c Records: 42
2026-09-03 17:54:50,144 [INFO] Unique in Metazoa_External_ID__c: 42
2026-09-03 17:54:50,147 [INFO] Preparing to upsert 42 records to Special_Conditions__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Application_Number__c', 'Comments__c', 'Due_Date__c', 'EDP_External_Id__c', 'End_Date__c', 'Special_Conditions_Description__c', 'Special_Conditions_Type__c', 'Start_Date__c', 'Status__c', 'Adviser__c']
2026-09-03 17:55:01,186 [INFO] Upsert complete. Time taken: 11.04s
2026-09-03 17:55:01,187 [INFO] Success: 36, Failures: 6
2026-09-03 17:55:01,187 [WARNING] Failures: [{'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'REQUIRED_FIELD_MISSING', 'message': 'Required fields are missing: [Due_Date__c]', 'fields': ['Due_Date__c']}]}, {'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'REQUIRED_FIELD_MISSING', 'mess

## 20 After User - Target__c

In [27]:
"""
reference_to
["Group", "User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Target-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= TARGET_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Target__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Target__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:56:03,230 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:56:03,231 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:56:03,233 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:56:03,233 INFO sqlalchemy.engine.Engine [generated in 0.00075s] ()
2026-09-03 17:56:03,236 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:56:03,237 INFO sqlalchemy.engine.Engine [generated in 0.00103s] ()
2026-09-03 17:56:03,239 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:56:03,239 INFO sqlalchemy.engine.Engine [generated in 0.00084s] ()
2026-09-03 17:56:03,248 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:56:03,250 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Target-ALL]
;
2026-09-03 17:56:03,252 INFO sqlalchemy.engine.Engin

2026-09-03 17:56:03,494 [INFO] Total Target__c Records: 3191
2026-09-03 17:56:03,495 [INFO] Unique in Metazoa_External_ID__c: 3191
2026-09-03 17:56:03,540 [INFO] Preparing to upsert 3191 records to Target__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Actual__c', 'End_Date__c', 'Name', 'Points_Achieved_SFG__c', 'Shading__c', 'Shading_Comments__c', 'Start_date__c', 'Target__c', 'User__c']
2026-09-03 17:56:09,739 [INFO] Upsert complete. Time taken: 6.20s
2026-09-03 17:56:09,741 [INFO] Success: 3191, Failures: 0
2026-09-03 17:56:09,750 [INFO] Load Results:       success  created                  id errors
0        True     True  a1kBm0000542GLaIAM     []
1        True     True  a1kBm0000542GLbIAM     []
2        True     True  a1kBm0000542GLcIAM     []
3        True     True  a1kBm0000542GLdIAM     []
4        True     True  a1kBm0000542GLeIAM     []
...       ...      ...                 ...    ...
3186     True

## 22 After User - Tracking__c

In [28]:
"""
reference_to
["RecordType"]
["Target__c"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Tracking-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= TRACKING_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Tracking__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Tracking__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 17:56:48,126 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 17:56:48,126 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 17:56:48,126 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 17:56:48,130 INFO sqlalchemy.engine.Engine [generated in 0.00079s] ()
2026-09-03 17:56:48,134 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 17:56:48,135 INFO sqlalchemy.engine.Engine [generated in 0.00129s] ()
2026-09-03 17:56:48,137 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 17:56:48,138 INFO sqlalchemy.engine.Engine [generated in 0.00090s] ()
2026-09-03 17:56:48,140 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 17:56:48,142 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Tracking-ALL]
;
2026-09-03 17:56:48,142 INFO sqlalchemy.engine.Eng

2026-09-03 17:56:49,690 [INFO] Total Tracking__c Records: 23024
2026-09-03 17:56:49,704 [INFO] Unique in Metazoa_External_ID__c: 23024
2026-09-03 17:56:50,059 [INFO] Preparing to upsert 23024 records to Tracking__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Comments__c', 'Date__c', 'End_Date__c', 'Hrs__c', 'Start_date__c', 'TrackingUniqueKey__c', 'Type__c', 'RecordTypeId', 'Target__c']
2026-09-03 17:59:39,557 [INFO] Upsert complete. Time taken: 169.50s
2026-09-03 17:59:39,560 [INFO] Success: 23024, Failures: 0
2026-09-03 17:59:39,605 [INFO] Load Results:        success  created                  id errors
0         True     True  a1nBm00000iied7IAA     []
1         True     True  a1nBm00000iied8IAA     []
2         True     True  a1nBm00000iied9IAA     []
3         True     True  a1nBm00000iiedAIAQ     []
4         True     True  a1nBm00000iiedBIAQ     []
...        ...      ...                 ...    ...
23019     True 

## 24 After User - Campaign

In [29]:
"""
reference_to
["Campaign"]
["RecordType"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Campaign-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= Campaign_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Campaign Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Campaign', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Breach_Reporting__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:00:49,741 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:00:49,742 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:00:49,742 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:00:49,742 INFO sqlalchemy.engine.Engine [generated in 0.00090s] ()
2026-09-03 18:00:49,749 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:00:49,750 INFO sqlalchemy.engine.Engine [generated in 0.00170s] ()
2026-09-03 18:00:49,752 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:00:49,753 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ()
2026-09-03 18:00:49,755 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:00:49,757 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Campaign-ALL]
;
2026-09-03 18:00:49,758 INFO sqlalchemy.engine.Eng

2026-09-03 18:00:49,918 [INFO] Total Campaign Records: 609
2026-09-03 18:00:49,920 [INFO] Unique in Metazoa_External_ID__c : 609
2026-09-03 18:00:49,944 [INFO] Preparing to upsert 609 records to Campaign with fields ['Metazoa_External_ID__c', 'ActualCost', 'BudgetedCost', 'Description', 'Duration__c', 'EndDate', 'ExpectedResponse', 'ExpectedRevenue', 'IsActive', 'Location__c', 'Name', 'NPS__c', 'NumberSent', 'Reason_for_training__c', 'StartDate', 'Status', 'Topic_Area__c', 'Total_Bridges_Attendees__c', 'Total_Consultum_Attendees__c', 'Total_FSP_Attendees__c', 'Total_GPG_Attendees__c', 'Total_Internal_Attendees__c', 'Total_Lonsdale_Attendees__c', 'Total_M3_Attendees__c', 'Total_PY_Attendees__c', 'Total_RI_Attendees__c', 'Total_Shadforth_Attendees__c', 'Total_TenFifty_Attendees__c', 'Training_mode_Method_of_contact__c', 'Training_Start_Date__c', 'Type', 'ParentId', 'CampaignMemberRecordTypeId', 'RecordTypeId']
2026-09-03 18:01:02,387 [INFO] Upsert complete. Time taken: 12.44s
2026-09-03 

## 25 After User - CampaignMember

In [30]:
"""
reference_to
["Account", "Contact", "Lead"]
["Campaign"]
["Contact"]
["Group", "User"]
["Lead"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CampaignMember-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= CampaignMember_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total CampaignMember Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CampaignMember', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Breach_Reporting__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:02:09,537 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:02:09,538 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:02:09,541 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:02:09,542 INFO sqlalchemy.engine.Engine [generated in 0.00099s] ()
2026-09-03 18:02:09,544 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:02:09,545 INFO sqlalchemy.engine.Engine [generated in 0.00088s] ()


2026-09-03 18:02:09,547 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:02:09,548 INFO sqlalchemy.engine.Engine [generated in 0.00101s] ()
2026-09-03 18:02:09,551 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:02:09,552 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_CampaignMember-ALL]
;
2026-09-03 18:02:09,553 INFO sqlalchemy.engine.Engine [generated in 0.00149s] ()
2026-09-03 18:02:09,943 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 18:02:09,943 [INFO] Total CampaignMember Records: 93
2026-09-03 18:02:09,953 [INFO] Preparing to upsert 93 records to CampaignMember with fields ['CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Licensee_text__c', 'Status', 'CampaignId', 'ContactId', 'LeadId']
2026-09-03 18:02:17,029 [INFO] Upsert complete. Time taken: 7.07s
2026-09-03 18:02:17,030 [INFO] Success: 93, Failures: 0
2026-09-03 18:02:17,035 [INFO] Load Results:     success  created                  id errors
0      True     True  00vBm00000A4GYHIA3     []
1      True     True  00vBm00000A4GYIIA3     []
2      True     True  00vBm00000A4GYJIA3     []
3      True     True  00vBm00000A4GYKIA3     []
4      True     True  00vBm00000A4GYLIA3     []
..      ...      ...                 ...    ...
88     True     True  00vBm00000A4GZhIAN     []
89     True     True  00vBm00000A4GZiIAN     []
90     True     True  00vBm00000A4GZjIAN     []
91     True     True  00vBm00000A4GZkIAN     []
92     True

## 25 After Campaign - Case_Snapshot__c

In [31]:
"""
reference_to
["Account"]
["Campaign"]
["Contact"]
["Contact"]
["Contract"]
["OpportunityHistory"]
["OpportunityHistory"]
["Pricebook2"]
["Quote"]
["RecordType"]
["User"]
"""
"""
object_api_name
Approval__c
Case
EmailMessage
Event
EventRelation
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Case_Snapshot-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= CASE_SNAPSHOT_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Case_Snapshot__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case_Snapshot__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Assurance_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:03:00,675 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:03:00,676 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:03:00,679 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:03:00,679 INFO sqlalchemy.engine.Engine [generated in 0.00084s] ()
2026-09-03 18:03:00,684 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:03:00,685 INFO sqlalchemy.engine.Engine [generated in 0.00172s] ()
2026-09-03 18:03:00,685 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:03:00,685 INFO sqlalchemy.engine.Engine [generated in 0.00121s] ()
2026-09-03 18:03:00,685 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:03:00,685 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Case_Snapshot-ALL]
;
2026-09-03 18:03:00,693 INFO sqlalchemy.engin

2026-09-03 18:03:00,939 [INFO] Total Case_Snapshot__c Records: 0
2026-09-03 18:03:00,940 [INFO] Unique in Metazoa_External_ID__c: 0
2026-09-03 18:03:00,942 [INFO] Preparing to upsert 0 records to Case_Snapshot__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Case_Type__c', 'EDP_External_Id__c', 'Schedule_Date__c', 'Schedule_Type__c', 'Status__c', 'Case__c']
2026-09-03 18:03:00,943 [ERROR] Error during upsert: data should not be empty for insert


ValueError: data should not be empty for insert

## 25 After User - Client_Service_Model_Implementation__c

In [32]:
"""
reference_to
["Account"]
["Contact"]
["RecordType"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
InitiativeChecklist__c
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Client_Service_Model_Implementation-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= Client_Service_Model_Implementation_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Client_Service_Model_Implementation__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:03:19,839 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:03:19,840 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:03:19,842 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:03:19,843 INFO sqlalchemy.engine.Engine [generated in 0.00111s] ()
2026-09-03 18:03:19,845 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:03:19,846 INFO sqlalchemy.engine.Engine [generated in 0.00069s] ()
2026-09-03 18:03:19,848 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:03:19,849 INFO sqlalchemy.engine.Engine [generated in 0.00113s] ()
2026-09-03 18:03:19,851 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:03:19,852 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Client_Service_Model_Implementation-ALL]
;
2026-09-03 18:03:19,853

2026-09-03 18:03:20,112 [INFO] Total Client_Service_Model_Implementation__c Records: 321
2026-09-03 18:03:20,113 [INFO] Unique in Metazoa_External_ID__c: 321
2026-09-03 18:03:20,152 [INFO] Preparing to upsert 321 records to Client_Service_Model_Implementation__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'ACR_Go_Live__c', 'ACR_Initiative_Comments__c', 'ACR_Question_1__c', 'ACR_Question_2__c', 'ACR_Question_3__c', 'ACR_Question_4__c', 'ACR_Question_5__c', 'Additional_Support__c', 'Advice_Case_Framework_Comments__c', 'Answer_1__c', 'Answer_2__c', 'Answer_3__c', 'Answer_4__c', 'Answer_5__c', 'CKRI_A9__c', 'CKRI_NotUmlaut_Comments__c', 'CKRI_Q7__c', 'CKRI_Q8__c', 'CKRI_Q9__c', 'CombinationBanner__c', 'Comments_for_Combination__c', 'Completion_Date__c', 'CRKI_A1__c', 'CRKI_A10__c', 'CRKI_A11__c', 'CRKI_A3__c', 'CRKI_A4__c', 'CRKI_A5_c__c', 'CRKI_A6__c', 'CRKI_A7__c', 'CRKI_Answer_0__c', 'CRKI_Comments_A1__c', 'CRKI_Q1__c', 'C

## 25 After User - LookupName__c

In [33]:
"""
reference_to
["Group", "User"]
["RecordType"]
["User"]
"""
"""
object_api_name
Case
EventRelation
InitiativeChecklist__c
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_LookupName-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= LOOKUPNAME_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total LookupName__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='LookupName__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:04:01,322 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:04:01,323 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:04:01,326 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:04:01,327 INFO sqlalchemy.engine.Engine [generated in 0.00085s] ()
2026-09-03 18:04:01,335 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:04:01,335 INFO sqlalchemy.engine.Engine [generated in 0.00089s] ()
2026-09-03 18:04:01,343 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:04:01,344 INFO sqlalchemy.engine.Engine [generated in 0.00086s] ()
2026-09-03 18:04:01,351 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:04:01,352 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_LookupName-ALL]
;
2026-09-03 18:04:01,353 INFO sqlalchemy.engine.E

2026-09-03 18:04:01,913 [INFO] Total LookupName__c Records: 2531
2026-09-03 18:04:01,916 [INFO] Unique in Metazoa_External_ID__c: 2531
2026-09-03 18:04:01,965 [INFO] Preparing to upsert 2531 records to LookupName__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedDate', 'CreatedById', 'LastModifiedDate', 'LastModifiedById', 'Action_Required__c', 'Issue_Identified__c', 'Name', 'Remediation_Timeframe__c', 'Security_Measure__c', 'SecurityCode__c', 'RecordTypeId']
2026-09-03 18:04:08,168 [INFO] Upsert complete. Time taken: 6.20s
2026-09-03 18:04:08,169 [INFO] Success: 2531, Failures: 0
2026-09-03 18:04:08,178 [INFO] Load Results:       success  created                  id errors
0        True     True  a1DBm00000DH7f7MAD     []
1        True     True  a1DBm00000DH7f8MAD     []
2        True     True  a1DBm00000DH7f9MAD     []
3        True     True  a1DBm00000DH7fAMAT     []
4        True     True  a1DBm00000DH7fBMAT     []
...       ...      ...                 ...    ...
2526  

## 25 After User - Product2

In [34]:
"""
reference_to
["ExternalDataSource"]
["User"]
"""
"""
object_api_name
Accreditations__c
Area_of_Advice__c
Asset
Case
EmailMessage
Event
EventRelation
Incident_File_Review__c
Product_Questions__c
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Product2-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= PRODUCT2_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Product2 Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Product2', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:04:39,511 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:04:39,512 INFO sqlalchemy.engine.Engine [raw sql] ()


2026-09-03 18:04:39,528 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:04:39,529 INFO sqlalchemy.engine.Engine [generated in 0.00116s] ()
2026-09-03 18:04:39,546 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:04:39,547 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()
2026-09-03 18:04:39,552 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:04:39,553 INFO sqlalchemy.engine.Engine [generated in 0.00084s] ()
2026-09-03 18:04:39,564 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:04:39,565 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Product2-ALL]
;
2026-09-03 18:04:39,566 INFO sqlalchemy.engine.Engine [generated in 0.00130s] ()
2026-09-03 18:04:39,671 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 18:04:39,678 [INFO] Total Product2 Records: 1
2026-09-03 18:04:39,679 [INFO] Unique in Metazoa_External_ID__c: 1
2026-09-03 18:04:39,682 [INFO] Preparing to upsert 1 records to Product2 with fields ['Metazoa_External_ID__c', 'Description', 'DisplayUrl', 'EDP_External_Id__c', 'ExternalId', 'Family', 'IsActive', 'Licensee_Groups_Permitted__c', 'Name', 'ProductCode', 'QuantityUnitOfMeasure', 'StockKeepingUnit', 'ExternalDataSourceId']
2026-09-03 18:04:46,636 [INFO] Upsert complete. Time taken: 6.95s
2026-09-03 18:04:46,637 [INFO] Success: 1, Failures: 0
2026-09-03 18:04:46,641 [INFO] Load Results:    success  created                  id errors
0     True     True  01tBm00000J17MHIAZ     []


## 26 After User - Accreditations__c

In [35]:
"""
reference_to
["Contact"]
["Product2"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Accreditations-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= ACCREDITATIONS_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Accreditations__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Accreditations__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:05:15,116 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:05:15,116 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:05:15,121 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:05:15,122 INFO sqlalchemy.engine.Engine [generated in 0.00162s] ()
2026-09-03 18:05:15,132 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:05:15,133 INFO sqlalchemy.engine.Engine [generated in 0.00086s] ()
2026-09-03 18:05:15,136 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:05:15,138 INFO sqlalchemy.engine.Engine [generated in 0.00098s] ()
2026-09-03 18:05:15,143 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:05:15,143 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Accreditations-ALL]
;
2026-09-03 18:05:15,143 INFO sqlalchemy.engi

2026-09-03 18:05:16,077 [INFO] Total Accreditations__c Records: 3227
2026-09-03 18:05:16,085 [INFO] Unique in Metazoa_External_ID__c: 3227
2026-09-03 18:05:16,155 [INFO] Preparing to upsert 3227 records to Accreditations__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Document_Received_Date__c', 'EDP_External_Id__c', 'Kaplan_Training_Date__c', 'Product_Family__c', 'Revoke_Date__c', 'Start_Date__c', 'Adviser__c', 'AccreditationType__c']
2026-09-03 18:05:22,425 [INFO] Upsert complete. Time taken: 6.27s
2026-09-03 18:05:22,426 [INFO] Success: 3227, Failures: 0
2026-09-03 18:05:22,436 [INFO] Load Results:       success  created                  id errors
0        True     True  a0ZBm000008MlbZMAS     []
1        True     True  a0ZBm000008MlbaMAC     []
2        True     True  a0ZBm000008MlbbMAC     []
3        True     True  a0ZBm000008MlbcMAC     []
4        True     True  a0ZBm000008MlbdMAC     []
...       ...      ...    

## 25 After User - Registered_Training_Organisation__c

In [36]:
"""
reference_to
["Group", "User"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
Qualification__c
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Registered_Training_Organisation-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= REGISTERED_TRAINING_ORGANIZATION_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Registered_Training_Organisation__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Registered_Training_Organisation__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:05:58,162 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:05:58,163 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:05:58,166 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:05:58,166 INFO sqlalchemy.engine.Engine [generated in 0.00100s] ()
2026-09-03 18:05:58,169 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:05:58,170 INFO sqlalchemy.engine.Engine [generated in 0.00084s] ()
2026-09-03 18:05:58,172 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:05:58,173 INFO sqlalchemy.engine.Engine [generated in 0.00096s] ()
2026-09-03 18:05:58,176 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:05:58,177 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Registered_Training_Organisation-ALL]
;
2026-09-03 18:05:58,178 IN

2026-09-03 18:05:58,244 [INFO] Total Registered_Training_Organisation__c Records: 72
2026-09-03 18:05:58,246 [INFO] Unique in Metazoa_External_ID__c: 72
2026-09-03 18:05:58,249 [INFO] Preparing to upsert 72 records to Registered_Training_Organisation__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedDate', 'CreatedById', 'LastModifiedDate', 'LastModifiedById', 'EDP_External_Id__c', 'Name']
2026-09-03 18:06:09,184 [INFO] Upsert complete. Time taken: 10.93s
2026-09-03 18:06:09,184 [INFO] Success: 72, Failures: 0
2026-09-03 18:06:09,189 [INFO] Load Results:     success  created                  id errors
0      True     True  a1bBm000005gKPFIA2     []
1      True     True  a1bBm000005gKPGIA2     []
2      True     True  a1bBm000005gKPHIA2     []
3      True     True  a1bBm000005gKPIIA2     []
4      True     True  a1bBm000005gKPJIA2     []
..      ...      ...                 ...    ...
67     True     True  a1bBm000005gKQKIA2     []
68     True     True  a1bBm000005gKQLIA2    

## 25 After User - BID_Case__c

In [37]:
"""
reference_to
["Contact"]
["Group", "User"]
["User"]
"""
"""
object_api_name
EventRelation
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_BID_Case-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= BID_CASE_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total BID_Case__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='BID_Case__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:06:38,447 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:06:38,448 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:06:38,451 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:06:38,452 INFO sqlalchemy.engine.Engine [generated in 0.00107s] ()
2026-09-03 18:06:38,455 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:06:38,455 INFO sqlalchemy.engine.Engine [generated in 0.00082s] ()
2026-09-03 18:06:38,457 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:06:38,458 INFO sqlalchemy.engine.Engine [generated in 0.00077s] ()
2026-09-03 18:06:38,460 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:06:38,461 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_BID_Case-ALL]
;
2026-09-03 18:06:38,462 INFO sqlalchemy.engine.Eng

2026-09-03 18:06:38,851 [INFO] Total BID_Case__c Records: 3708
2026-09-03 18:06:38,854 [INFO] Unique in Metazoa_External_ID__c: 3708
2026-09-03 18:06:38,968 [INFO] Preparing to upsert 3708 records to BID_Case__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedDate', 'CreatedById', 'LastModifiedDate', 'LastModifiedById', 'Adviser_ph_call_completed__c', 'Area_of_advice__c', 'Call_invoiced_to_client__c', 'Case_issue__c', 'Case_Status__c', 'Complexity_of_Advice__c', 'Customer_1_First_Name__c', 'Customer_1_Surname__c', 'Customer_2_First_Name__c', 'Customer_2_Surname__c', 'Date_BID_review_completed__c', 'Date_call_completed__c', 'Date_case_due__c', 'Date_of_BID_review_received__c', 'Date_scheduled_audit__c', 'Days_overdue__c', 'Initial_triage_outcome__c', 'Integration_ID__c', 'Invoiced_to_the_client__c', 'IsActive__c', 'Name', 'No_of_new_documents_received_01__c', 'No_of_new_documents_received_02__c', 'Outcome_of_BID_review__c', 'Resub01_date_completed__c', 'Resub01_date_received__

## 25 After User - Question_Group__c

In [38]:
"""
reference_to
["Group", "User"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
Master_Questions__c
MasterQuestions_Targetedreviews__c
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Question_Group-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= QUESTION_GROUP_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Question_Group__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Group__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:07:16,983 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:07:16,985 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:07:16,987 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:07:16,988 INFO sqlalchemy.engine.Engine [generated in 0.00091s] ()
2026-09-03 18:07:16,991 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:07:16,992 INFO sqlalchemy.engine.Engine [generated in 0.00141s] ()
2026-09-03 18:07:16,994 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:07:16,995 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()
2026-09-03 18:07:16,997 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:07:16,998 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Question_Group-ALL]
;
2026-09-03 18:07:16,999 INFO sqlalchemy.engi

2026-09-03 18:07:17,071 [INFO] Total Question_Group__c Records: 67
2026-09-03 18:07:17,072 [INFO] Unique in Id: 67
2026-09-03 18:07:17,076 [INFO] Preparing to upsert 67 records to Question_Group__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Description__c', 'EDP_External_Id__c', 'External_ID__c', 'Group_Number__c', 'Name', 'Questionnaire__c', 'Section_Title__c']
2026-09-03 18:07:23,003 [INFO] Upsert complete. Time taken: 5.93s
2026-09-03 18:07:23,003 [INFO] Success: 67, Failures: 0
2026-09-03 18:07:23,008 [INFO] Load Results:     success  created                  id errors
0      True     True  a1WBm000004TTp7MAG     []
1      True     True  a1WBm000004TTp8MAG     []
2      True     True  a1WBm000004TTp9MAG     []
3      True     True  a1WBm000004TTpAMAW     []
4      True     True  a1WBm000004TTpBMAW     []
..      ...      ...                 ...    ...
62     True     True  a1WBm000004TTq7MAG     []
63    

## 20 After User - Master_Questions__c

In [40]:
"""
reference_to
["Group", "User"]
["Question_Group__c"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
Library_Text__c
Product_Questions__c
Question_Responses__c
Response_Finding__c
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Questions-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= MASTER_QUESTIONS_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

# df_transformed

logger.info(f"Total Master_Questions__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Master_Questions__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:45:39,730 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:45:39,730 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:45:39,730 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:45:39,730 INFO sqlalchemy.engine.Engine [generated in 0.00102s] ()
2026-09-03 18:45:39,730 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:45:39,730 INFO sqlalchemy.engine.Engine [generated in 0.00101s] ()
2026-09-03 18:45:39,730 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:45:39,730 INFO sqlalchemy.engine.Engine [generated in 0.00103s] ()
2026-09-03 18:45:39,745 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:45:39,745 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Questions-ALL]
;
2026-09-03 18:45:39,745 INFO sqlalchemy.en

2026-09-03 18:45:39,848 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 18:45:39,852 [INFO] Total Master_Questions__c Records: 393
2026-09-03 18:45:39,854 [INFO] Unique in Id: 393
2026-09-03 18:45:39,877 [INFO] Preparing to upsert 393 records to Master_Questions__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Action__c', 'Actions_Taken__c', 'Active_From__c', 'Active_Until__c', 'Comment__c', 'Default_Issue_Text_for_Vetting__c', 'Default_Remedial_Actions_for_Vetting__c', 'Description__c', 'Display_Text_For_Reports__c', 'EDP_External_Id__c', 'External_ID__c', 'Impact_on_Score__c', 'Incident_Remediation_Text__c', 'Key_Area__c', 'Points__c', 'Priority__c', 'Process_Area__c', 'Question_Number__c', 'Question_Text__c', 'Reportable__c', 'Reportable_For_Reviews__c', 'Response_Options__c', 'Rootcause_Options__c', 'Score__c', 'Standard__c', 'Vetting_Action__c', 'Vetting_Comments__c', 'Vetting_Mandatory_Question__c', 'Wave_Internal__c', 'Question_Group__c']
2026-09-03 18:45:46,260 [I

## 27 After Question_Group__c - MasterQuestions_TargetedReviews__c

In [41]:
"""
reference_to
["Group", "User"]
["Question_Group__c"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
Library_Text__c
Question_Responses__c
Response_Finding__c
SelectedQuestions_Targetedreviews__c
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_MasterQuestions_Targetedreviews-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= MASTERQUESTIONS_TARGETEDREVIEWS_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total MasterQuestions_Targetedreviews__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='MasterQuestions_Targetedreviews__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:46:56,982 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:46:56,982 INFO sqlalchemy.engine.Engine [raw sql] ()


2026-09-03 18:46:56,985 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:46:56,986 INFO sqlalchemy.engine.Engine [generated in 0.00145s] ()
2026-09-03 18:46:56,989 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:46:56,990 INFO sqlalchemy.engine.Engine [generated in 0.00102s] ()
2026-09-03 18:46:56,992 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:46:56,992 INFO sqlalchemy.engine.Engine [generated in 0.00083s] ()
2026-09-03 18:46:56,994 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:46:56,997 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_MasterQuestions_Targetedreviews-ALL]
;
2026-09-03 18:46:57,000 INFO sqlalchemy.engine.Engine [generated in 0.00410s] ()
2026-09-03 18:46:57,094 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 18:46:57,097 [INFO] Total MasterQuestions_Targetedreviews__c Records: 133
2026-09-03 18:46:57,099 [INFO] Unique in Id: 133
2026-09-03 18:46:57,106 [INFO] Preparing to upsert 133 records to MasterQuestions_Targetedreviews__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Action__c', 'Actions_Taken__c', 'Active_From__c', 'Active_Until__c', 'Comment__c', 'Default_Issue_Text_for_Vetting__c', 'Default_Remedial_Actions_for_Vetting__c', 'Description__c', 'Display_Text_For_Reports__c', 'EDP_External_Id__c', 'External_ID__c', 'Impact_on_Score__c', 'Incident_Remediation_Text__c', 'Is_Bank_Question__c', 'Key_Area__c', 'Points__c', 'Priority__c', 'Process_Area__c', 'Qn_Text_Key__c', 'Question_Category__c', 'Question_Number__c', 'Question_Text__c', 'Reportable__c', 'Reportable_For_Reviews__c', 'Response_Options__c', 'Rootcause_Options__c', 'Standard__c', 'Vetting_Action__c', 'Vetting_Comments__c', 'Vetting_Mandator

## 29 After Master_Questions__c - Library_Text__c

In [42]:
"""
reference_to
["Group", "User"]
["Master_Questions__c"]
["MasterQuestions_Targetedreviews__c"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Library_Text-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= LIBRARY_TEXT_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Library_Text__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Library_Text__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:48:18,464 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:48:18,465 INFO sqlalchemy.engine.Engine [raw sql] ()


2026-09-03 18:48:18,468 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:48:18,468 INFO sqlalchemy.engine.Engine [generated in 0.00073s] ()
2026-09-03 18:48:18,471 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:48:18,472 INFO sqlalchemy.engine.Engine [generated in 0.00077s] ()
2026-09-03 18:48:18,474 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:48:18,475 INFO sqlalchemy.engine.Engine [generated in 0.00071s] ()
2026-09-03 18:48:18,477 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:48:18,478 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Library_Text-ALL]
;
2026-09-03 18:48:18,479 INFO sqlalchemy.engine.Engine [generated in 0.00137s] ()
2026-09-03 18:48:18,693 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 18:48:18,699 [INFO] Total Library_Text__c Records: 710
2026-09-03 18:48:18,700 [INFO] Unique in Metazoa_External_ID__c: 710
2026-09-03 18:48:18,711 [INFO] Preparing to upsert 710 records to Library_Text__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Actions_Required__c', 'Expiry_Date__c', 'External_ID__c', 'Findings__c', 'Master_Question__c', 'Targeted_Assurance_Question_Bank__c']
2026-09-03 18:48:24,817 [INFO] Upsert complete. Time taken: 6.11s
2026-09-03 18:48:24,818 [INFO] Success: 710, Failures: 0
2026-09-03 18:48:24,827 [INFO] Load Results:      success  created                  id errors
0       True     True  a1CBm00000cGkYsMAK     []
1       True     True  a1CBm00000cGkYtMAK     []
2       True     True  a1CBm00000cGkYuMAK     []
3       True     True  a1CBm00000cGkYvMAK     []
4       True     True  a1CBm00000cGkYwMAK     []
..       ...      ...                 ...    ...
705     True     

## 25 After User - Opportunity

In [43]:
"""
reference_to
["Account"]
["Campaign"]
["Contact"]
["Contract"]
["OpportunityHistory"]
["Pricebook2"]
["Quote"]
["RecordType"]
["User"]
"""
"""
object_api_name
Approval__c
Case
EmailMessage
Event
EventRelation
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Opportunity-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields = OPPORTUNITY_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Opportunity Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Opportunity', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:49:13,895 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 18:49:13,896 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 18:49:13,898 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 18:49:13,899 INFO sqlalchemy.engine.Engine [generated in 0.00094s] ()
2026-09-03 18:49:13,901 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 18:49:13,903 INFO sqlalchemy.engine.Engine [generated in 0.00109s] ()
2026-09-03 18:49:13,905 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 18:49:13,906 INFO sqlalchemy.engine.Engine [generated in 0.00075s] ()
2026-09-03 18:49:13,908 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 18:49:13,909 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Opportunity-ALL]
;
2026-09-03 18:49:13,910 INFO sqlalchemy.engine.

2026-09-03 18:49:14,313 [INFO] Total Opportunity Records: 559
2026-09-03 18:49:14,313 [INFO] Unique in Metazoa_External_ID__c: 559
2026-09-03 18:49:14,417 [INFO] Preparing to upsert 559 records to Opportunity with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedDate', 'CreatedById', 'LastModifiedDate', 'LastModifiedById', 'AccountId', 'RecordTypeId', 'IsPrivate', 'Name', 'StageName', 'Amount', 'TotalOpportunityQuantity', 'CloseDate', 'NextStep', 'LeadSource', 'ForecastCategoryName', 'Pricebook2Id', 'Budget_Confirmed__c', 'Discovery_Completed__c', 'ROI_Analysis_Completed__c', 'AFP_Police_Check_Date__c', 'Loss_Reason__c', 'AFP_Police_Check__c', 'ARC_Login_Cancelled_Date__c', 'ARC_login_cancelled__c', 'AR_CAR_Agreements_Scanned_Saved_Date__c', 'AR_CAR_Agreements_Scanned_Saved__c', 'AR_CAR_Agreements_issued_Date__c', 'AR_CAR_Agreements_issued__c', 'AR_CAR_Agreements_received_Date__c', 'AR_CAR_Agreements_received__c', 'ASIC_Ceased_Date__c', 'ASIC_Ceased__c', 'ASIC_Notification_Date__c'

## 27 After Opportunity - Case - Run 1 - INSERT without ParentId and Escalation_Case__c

In [47]:
"""
reference_to
["Account"]
["Asset"]
["BusinessHours"]
["Case"]
["Contact"]
["EmailMessage", "MessagingSession", "SocialPost"]
["Entitlement"]
["Group", "User"]
["LookupName__c"]
["Opportunity"]
["Product2"]
["RecordType"]
["ServiceContract"]
["User"]
"""
"""
object_api_name
Case
Case_Snapshot__c
CaseComment
Complained_Advisers__c
EmailMessage
EmailMessage
Event
EventRelation
Refund_Request__c
Snapshot_Data__c
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Case-ALL] where [Id] = '' and [Practice_Name__c] not like '%+' and [ContactId] not like '%+'

;"""

from sf_target_fields import *
ls_tgt_objFields= CASE_TARGET_FIELDS
ls_tgt_objFields = [x for x in ls_tgt_objFields if x not in ('ParentId','Escalation_Case__c')]

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

2026-09-03 19:06:33,135 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:06:33,136 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:06:33,138 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:06:33,139 INFO sqlalchemy.engine.Engine [generated in 0.00107s] ()
2026-09-03 19:06:33,141 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:06:33,142 INFO sqlalchemy.engine.Engine [generated in 0.00107s] ()
2026-09-03 19:06:33,144 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:06:33,145 INFO sqlalchemy.engine.Engine [generated in 0.00079s] ()


2026-09-03 19:06:33,147 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:06:33,149 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Case-ALL] where [Id] = '' and [Practice_Name__c] not like '%+' and [ContactId] not like '%+'

;
2026-09-03 19:06:33,151 INFO sqlalchemy.engine.Engine [generated in 0.00131s] ()
2026-09-03 19:06:35,692 INFO sqlalchemy.engine.Engine COMMIT


,Metazoa_External_ID__c,OwnerId,CreatedById,CreatedDate,LastModifiedById,LastModifiedDate,ABN_ACN_Check__c,ABN_ACN_Check_Date__c,AccountId,Additional_Directorships_Check__c,...,SuppliedName,SuppliedPhone,Tax_Practitioner_Board__c,Technical_Knowledge_Date__c,Technical_Knowledge_Test__c,TPB_Check_Date__c,Trust_Deed_Checks__c,Trust_Deed_Checks_Date__c,Type,Upcoming_Task_Due_Date__c
0,500Mq00000fRARzIAO,005W2000000K2tYIAS,005Bm00000Y8HZyIAN,2025-11-27T05:44:49.000+0000,005W2000000K2tYIAS,2026-07-16T05:41:40.000+0000,,None,001Bm000018tnbEIAQ,In progress,...,None,None,,None,Not Applicable,None,,None,,None
1,500Mq00000liCtrIAE,005W2000000K2tYIAS,005Bm00000Y8HZyIAN,2026-03-31T03:27:40.000+0000,005W2000000K2tYIAS,2026-07-26T23:58:56.000+0000,,None,001Bm000018tqiyIAA,In progress,...,None,None,,None,Not Applicable,None,,None,,None
2,500Mq00000niX7KIAU,005W2000000K2tYIAS,005Bm00000Y8HZyIAN,2026-05-07T22:59:42.000+0000,005W2000000K2tYIAS,2026-05-14T07:18:15.000+0000,,None,001Bm000018tqinIAA,In progress,...,None,None,,None,Not Applicable,None,,None,,None
3,500Mq00000ocDAqIAM,005W2000000K2tYIAS,005Bm00000Y8J71IAF,2026-05-24T23:48:28.000+0000,005W2000000K2pkIAC,2026-07-15T00:15:05.000+0000,,None,001Bm000018tnbEIAQ,In progress,...,None,None,,None,Not Applicable,None,,None,,None
4,500Mq00000pnF7CIAU,005W2000000K2tYIAS,005Bm00000Y8J71IAF,2026-06-12T02:37:05.000+0000,005W2000000K2tYIAS,2026-06-12T07:50:54.000+0000,,None,001Bm000018tqj3IAA,In progress,...,None,None,,None,Not Applicable,None,,None,,None
5,500Mq00000pnjoDIAQ,005W2000000K2tYIAS,005Bm00000Y8J71IAF,2026-06-12T02:44:33.000+0000,005W2000000K2pkIAC,2026-07-10T02:15:08.000+0000,,None,001Bm000018tqiiIAA,In progress,...,None,None,,None,Not Applicable,None,,None,,None
6,500Mq00000qCSHBIA4,005W2000000K2tYIAS,005Bm00000Y8HZyIAN,2026-06-18T23:35:18.000+0000,005W2000000K2tYIAS,2026-08-09T23:17:21.000+0000,,None,001Bm000018tnbXIAQ,In progress,...,None,None,,None,Not Applicable,None,,None,,None
7,500Mq00000qX5ayIAC,005W2000000K2tYIAS,005Bm00000Y8HZyIAN,2026-06-24T00:50:43.000+0000,005W2000000K2tYIAS,2026-08-19T00:53:07.000+0000,,None,001Bm000018tqinIAA,In progress,...,None,None,,None,Not Applicable,None,,None,,None
8,500Mq00000qXEpYIAW,005W2000000K2tYIAS,005Bm00000Y8HZyIAN,2026-06-24T01:27:34.000+0000,005W2000000K2tYIAS,2026-08-14T05:59:42.000+0000,,None,001Bm000018tqiiIAA,In progress,...,None,None,,None,Not Applicable,None,,None,,None
9,500Mq00000snV0LIAU,005W2000000K2tYIAS,005Bm00000Y8HZyIAN,2026-07-31T02:45:55.000+0000,005W2000000K2tYIAS,2026-08-14T10:11:48.000+0000,,None,001Bm000018tqivIAA,In progress,...,None,None,,None,Not Applicable,None,,None,,None


In [45]:

logger.info(f"Total Case Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=1000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 18:53:03,427 [INFO] Total Case Records: 7684
2026-09-03 18:53:03,433 [INFO] Unique in Metazoa_External_ID__c: 7684
2026-09-03 18:53:04,845 [INFO] Preparing to upsert 7684 records to Case with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'ABN_ACN_Check__c', 'ABN_ACN_Check_Date__c', 'AccountId', 'Additional_Directorships_Check__c', 'Additional_Directorships_Check_Date__c', 'ADG_ANZ_child__c', 'Amount_Paid_to_Client__c', 'Approval_Comments__c', 'Approvals_Professional_Standard__c', 'APRA_Check__c', 'APRA_Check_Date__c', 'AR_Register_Check__c', 'AR_Register_Check_Date__c', 'Area_Responsible__c', 'ASIC_Check__c', 'ASIC_Check_Date__c', 'ASIC_EU_Check__c', 'ASIC_EU_Check_Date__c', 'AssetId', 'Banned_Disqualified_Check__c', 'Banned_Disqualified_Check_Date__c', 'Business_insurance__c', 'Business_insurance_Completion_Date__c', 'BusinessHoursId', 'Case_Managed_by__c', 'Claimed_Amount__c', 'Client_1_First_Name__c', 'C

## 27 After Opportunity - Case - Run 2 - UPSERT with ParentId and Escalation_Case__c

In [46]:
"""
reference_to
["Account"]
["Asset"]
["BusinessHours"]
["Case"]
["Contact"]
["EmailMessage", "MessagingSession", "SocialPost"]
["Entitlement"]
["Group", "User"]
["LookupName__c"]
["Opportunity"]
["Product2"]
["RecordType"]
["ServiceContract"]
["User"]
"""
"""
object_api_name
Case
Case_Snapshot__c
CaseComment
Complained_Advisers__c
EmailMessage
EmailMessage
Event
EventRelation
Refund_Request__c
Snapshot_Data__c
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Case-ALL] where [ParentId] not like '%+' and [Escalation_Case__c] not like '%+'

;"""

ls_tgt_objFields = ['Id','ParentId','Escalation_Case__c']

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

2026-09-03 19:05:32,183 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:05:32,183 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:05:32,196 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:05:32,196 INFO sqlalchemy.engine.Engine [generated in 0.00106s] ()
2026-09-03 19:05:32,200 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:05:32,201 INFO sqlalchemy.engine.Engine [generated in 0.00128s] ()
2026-09-03 19:05:32,203 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:05:32,204 INFO sqlalchemy.engine.Engine [generated in 0.00082s] ()
2026-09-03 19:05:32,207 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:05:32,208 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Case-ALL] where [ParentId] not like '%+' and [Escalation_Case__c] 

,Id,ParentId,Escalation_Case__c


In [ ]:

logger.info(f"Total Case Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=1000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-01 17:04:04,572 [INFO] Total Case Records: 151
2026-09-01 17:04:04,574 [INFO] Unique in Metazoa_External_ID__c: 151
2026-09-01 17:04:04,626 [INFO] Preparing to upsert 151 records to Case with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'ABN_ACN_Check__c', 'ABN_ACN_Check_Date__c', 'AccountId', 'Additional_Directorships_Check__c', 'Additional_Directorships_Check_Date__c', 'ADG_ANZ_child__c', 'Amount_Paid_to_Client__c', 'Approval_Comments__c', 'Approvals_Professional_Standard__c', 'APRA_Check__c', 'APRA_Check_Date__c', 'AR_Register_Check__c', 'AR_Register_Check_Date__c', 'Area_Responsible__c', 'ASIC_Check__c', 'ASIC_Check_Date__c', 'ASIC_EU_Check__c', 'ASIC_EU_Check_Date__c', 'AssetId', 'Banned_Disqualified_Check__c', 'Banned_Disqualified_Check_Date__c', 'Business_insurance__c', 'Business_insurance_Completion_Date__c', 'BusinessHoursId', 'Case_Managed_by__c', 'Claimed_Amount__c', 'Client_1_First_Name__c', 'Clie

## 30 After User - Incident_File_Review__c

In [48]:
"""
reference_to
["Account"]
["Assurance_Review__c"]     -- circular reference
["Contact"]
["Contact"]
["Group", "User"]
["Product2"]
["User"]
"""
"""
object_api_name
Assurance_Event__c
Breach_Reporting__c
EmailMessage
Event
EventRelation
Question_Responses__c
Refund_Request__c
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Incident_File_Review-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= Incident_File_Review_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Incident_File_Review__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Incident_File_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=1000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Incident_File_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 19:06:59,353 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:06:59,356 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:06:59,358 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:06:59,359 INFO sqlalchemy.engine.Engine [generated in 0.00101s] ()


2026-09-03 19:06:59,362 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:06:59,364 INFO sqlalchemy.engine.Engine [generated in 0.00196s] ()
2026-09-03 19:06:59,366 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:06:59,367 INFO sqlalchemy.engine.Engine [generated in 0.00104s] ()
2026-09-03 19:06:59,369 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:06:59,370 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Incident_File_Review-ALL]
;
2026-09-03 19:06:59,371 INFO sqlalchemy.engine.Engine [generated in 0.00118s] ()
2026-09-03 19:07:00,748 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 19:07:00,795 [INFO] Total Incident_File_Review__c Records: 3507
2026-09-03 19:07:00,798 [INFO] Unique in Metazoa_External_ID__c: 3507
2026-09-03 19:07:01,107 [INFO] Preparing to upsert 3507 records to Incident_File_Review__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'A1__c', 'A2__c', 'A3__c', 'A4__c', 'A5__c', 'Account_Number__c', 'Actions_Completed_to_Date__c', 'Application_Date__c', 'ASIC_Notification__c', 'Client_Name__c', 'Close_Date__c', 'Comments__c', 'COR_ID__c', 'COR_Logged_Date__c', 'COR_Status__c', 'Date_Closed_in_COR__c', 'Date_Occurred__c', 'Date_of_ASIC_Notification__c', 'Date_of_Governance_Forum_Escalation__c', 'Date_of_Payment__c', 'Date_of_Referral__c', 'Date_PI_Insurer_Notified__c', 'Discovery_Date__c', 'EDP_External_Id__c', 'Event_Type__c', 'Fact_Find_Date__c', 'Fault_Area__c', 'Hidden_in_AdviserONE__c', 'Incident_Category_Field__c', 'Incident_Description__c', 'Incident_Title__c',

## 32 After Incident_File_Review__c - Assurance_Event__c

In [49]:
"""
reference_to
["Group", "User"]
["Incident_File_Review__c"]
["RecordType"]
["User"]
"""
"""
object_api_name
Assurance_Review__c
EventRelation
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Assurance_Event-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= ASSURANCE_EVENT_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Assurance_Event__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Assurance_Event__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_Service_Model_Implementation__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 19:08:40,645 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:08:40,645 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:08:40,653 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:08:40,654 INFO sqlalchemy.engine.Engine [generated in 0.00096s] ()
2026-09-03 19:08:40,657 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:08:40,658 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()
2026-09-03 19:08:40,660 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:08:40,660 INFO sqlalchemy.engine.Engine [generated in 0.00100s] ()
2026-09-03 19:08:40,663 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:08:40,664 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Assurance_Event-ALL]
;
2026-09-03 19:08:40,665 INFO sqlalchemy.eng

2026-09-03 19:08:40,836 [INFO] Total Assurance_Event__c Records: 9
2026-09-03 19:08:40,838 [INFO] Unique in Metazoa_External_ID__c: 9
2026-09-03 19:08:40,841 [INFO] Preparing to upsert 9 records to Assurance_Event__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Closed_Date__c', 'Comments__c', 'Event_Record_Sub_Type__c', 'Event_Record_Type__c', 'Event_Referral_Date__c', 'Event_Scope__c', 'Event_Scope_Comment__c', 'Event_Status__c', 'Number_Of_Client_Files__c', 'Number_Of_Unique_Advisers__c', 'Review_Count__c', 'Related_Incident__c', 'RecordTypeId', 'Event_Lead__c']
2026-09-03 19:08:41,719 [INFO] Upsert complete. Time taken: 0.88s
2026-09-03 19:08:41,722 [INFO] Success: 9, Failures: 0
2026-09-03 19:08:41,728 [INFO] Load Results:    success  created                  id errors
0     True     True  a0nBm0000029ESPIA2     []
1     True     True  a0nBm0000029ESQIA2     []
2     True     True  a0nBm0000029ESRIA2     []

## 40 After Incident_File_Review__c - Assurance_Review__c - Run 1 - INSERT, without Risk_Forum__c, Original_Request_Number__c, and Resubmission_case__c
### May need to execute this loop multiple times to clear circular references
1. Assurance_Review__c - INSERT without Risk_Forum__c, Original_Request_Number__c, and Resubmission_case__c
2. Consequence_Management_Forum__c - INSERT
3. Assurance_Review__c - UPSERT with Risk_Forum__c, Original_Request_Number__c, and Resubmission_case__c

In [50]:
"""
reference_to
["Account"]
["Assurance_Event__c"]
["Assurance_Review__c"]
["Assurance_Review__c"]
["Assurance_Review__c"]
["Consequence_Management_Forum_Entry__c"]
["Contact"]
["Contact"]
["Group", "User"]
["RecordType"]
["User"]
"""
"""
object_api_name
ASN_Indicator_History__c
Assurance_Review__c
Client_File__c
Consequence_Management_Forum_Entry__c
EmailMessage
Event
EventRelation
Incident_File_Review__c
QA__c
QA_Assignment_Hisotry__c
Question_Responses__c
Reviews_Accounted__c
SelectedQuestions_Targetedreviews__c
Task
TaskRelation
Vetting_Result__c
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Assurance_Review-ALL] where [Id] = ''
;"""

from sf_target_fields import *
ls_tgt_objFields= ASSURANCE_REVIEW_TARGET_FIELDS
# ls_tgt_objFields = [x for x in ls_tgt_objFields if x not in ('Review__c')]
ls_tgt_objFields = [x for x in ls_tgt_objFields if x not in ('Risk_Forum__c','Original_Request_Number__c','Resubmission_case__c')]

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed


2026-09-03 19:10:12,966 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:10:12,967 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:10:12,969 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:10:12,970 INFO sqlalchemy.engine.Engine [generated in 0.00088s] ()
2026-09-03 19:10:12,973 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:10:12,973 INFO sqlalchemy.engine.Engine [generated in 0.00067s] ()


2026-09-03 19:10:12,976 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:10:12,976 INFO sqlalchemy.engine.Engine [generated in 0.00111s] ()
2026-09-03 19:10:12,979 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:10:12,979 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Assurance_Review-ALL] where [Id] = ''
;
2026-09-03 19:10:12,979 INFO sqlalchemy.engine.Engine [generated in 0.00132s] ()
2026-09-03 19:10:16,594 INFO sqlalchemy.engine.Engine COMMIT


,Metazoa_External_ID__c,OwnerId,CreatedById,CreatedDate,LastModifiedById,LastModifiedDate,Actions_from_Vetting_Result__c,Advice_Document_Type__c,Advice_Document_Type_AdviserONE__c,Advice_Risk_Escalation_Comments__c,...,Vetting_Result__c,Workflow_Step__c,Practice__c,Assurance_Event__c,Cloned_Targeted_Review__c,Adviser__c,Adviser_AdviserONE__c,RecordTypeId,Assurance_Reviewer__c,Licensee_Supervisor_Name__c
0,a09Mq0000075jlFIAQ,005W2000000K2ryIAC,005W2000000K2hfIAC,2026-07-09T04:26:53.000+0000,005W2000000K2hfIAC,2026-07-09T04:26:54.000+0000,None,None,None,None,...,In Progress,Scheduled,001Bm000018tnbpIAA,,None,003Bm00001HKulRIAT,None,012W2000000RbEnIAK,None,005W2000000K2mBIAS
1,a09Mq0000079REfIAM,005W2000000K2ryIAC,005W2000000K2hfIAC,2026-07-13T04:34:02.000+0000,005W2000000K2hfIAC,2026-07-13T04:34:03.000+0000,None,None,None,None,...,In Progress,Scheduled,001Bm000018tnbpIAA,,None,003Bm00001HKujbIAD,None,012W2000000RbEnIAK,None,005W2000000K2mBIAS
2,a09Mq0000079TMvIAM,005W2000000K2ryIAC,005W2000000K2hfIAC,2026-07-13T04:55:34.000+0000,005W2000000K2hfIAC,2026-07-13T04:55:35.000+0000,None,None,None,None,...,In Progress,Scheduled,001Bm000018tnbpIAA,,None,003Bm00001HKukYIAT,None,012W2000000RbEnIAK,None,005W2000000K2mBIAS
3,a09Mq0000075VqLIAU,005W2000000K2ryIAC,005W2000000K2hfIAC,2026-07-09T01:30:44.000+0000,005W2000000K2hfIAC,2026-07-09T01:30:44.000+0000,None,None,None,None,...,In Progress,Scheduled,001Bm000018tnbpIAA,,None,003Bm00001HKukPIAT,None,012W2000000RbEnIAK,None,005W2000000K2mBIAS
4,a09Mq0000052aULIAY,005W2000000K2ryIAC,005W2000000K2hfIAC,2026-03-13T23:59:21.000+0000,005W2000000K2hfIAC,2026-07-07T00:47:49.000+0000,None,None,None,None,...,In Progress,Cancelled,001Bm000018tnbpIAA,,None,003Bm00001HKujfIAD,None,012W2000000RbEoIAK,None,005W2000000K2mBIAS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6309,a09Mq000003udY1IAI,005W2000000K2jVIAS,005W2000000K2hfIAC,2025-12-10T22:48:37.000+0000,005W2000000K2jVIAS,2026-08-24T21:18:18.000+0000,None,None,None,None,...,In Progress,Interim Report Sent,001Bm000018tnbpIAA,,None,003Bm00001HKujWIAT,003Bm00001HKujWIAT,012W2000000RbEnIAK,005W2000000K2jVIAS,005Bm00000Y8J7GIAV
6310,a09Mq000004g2eDIAQ,005W2000000K2jVIAS,005W2000000K2hfIAC,2026-02-13T04:32:48.000+0000,005W2000000K2jVIAS,2026-08-25T14:00:01.000+0000,None,None,None,None,...,In Progress,Interim Report Sent,001Bm000018tnbpIAA,,None,003Bm00001HKukYIAT,003Bm00001HKukYIAT,012W2000000RbEnIAK,005W2000000K2jVIAS,005Bm00000Y8J7GIAV
6311,a09Mq000004fxg9IAA,005W2000000K2plIAC,005W2000000K2hfIAC,2026-02-13T03:04:21.000+0000,005W2000000K2plIAC,2026-09-01T04:15:24.000+0000,None,None,None,None,...,In Progress,Draft Report Sent,001Bm000018tnbpIAA,,None,003Bm00001HKulRIAT,003Bm00001HKulRIAT,012W2000000RbEnIAK,005W2000000K2plIAC,005Bm00000Y8J7GIAV
6312,a09Mq000003qEhtIAE,005W2000000K2fNIAS,005W2000000K2hfIAC,2025-12-05T01:07:14.000+0000,005W2000000K2pkIAC,2026-08-18T18:00:05.000+0000,None,None,None,None,...,In Progress,Remedial Actions Completed,001Bm000018tnbpIAA,,None,003Bm00001HKutfIAD,003Bm00001HKutfIAD,012W2000000RbEoIAK,005W2000000K2fNIAS,005Bm00000Y8J7GIAV


In [51]:

logger.info(f"Total Assurance_Review__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Assurance_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=100, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Assurance_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 19:10:34,437 [INFO] Total Assurance_Review__c Records: 6314
2026-09-03 19:10:34,441 [INFO] Unique in Metazoa_External_ID__c: 6314
2026-09-03 19:10:35,189 [INFO] Preparing to upsert 6314 records to Assurance_Review__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Actions_from_Vetting_Result__c', 'Advice_Document_Type__c', 'Advice_Document_Type_AdviserONE__c', 'Advice_Risk_Escalation_Comments__c', 'Advice_Risk_Escalation_Recommendations__c', 'All_Response_Yes__c', 'Any_Outstanding_Remediation__c', 'Appeal_Closed_Date__c', 'Appeal_Received_Date__c', 'Appeal_Status__c', 'Appointment_Date__c', 'Appointment_Location__c', 'Appointment_Time__c', 'Are_all_Documents_Received__c', 'Areas_of_Advice_for_Vetting__c', 'ASN_Score__c', 'Atleast_Two_Potential_Repeat_Offence_s__c', 'Client_Name__c', 'ClientID_Xplan__c', 'Closed_Date__c', 'Comment__c', 'Comment1__c', 'Comment2__c', 'Comment3__c', 'Comment4__c', 'Comments

## 42 After Assurance_Review__ Run 1 - Consequence_Management_Forum_Entry__c

In [52]:
"""
reference_to
["Account"]
["Assurance_Review__c"]
["Contact"]
["Group", "User"]
["RecordType"]
["User"]
"""
"""
object_api_name
Assurance_Review__c
EmailMessage
Event
EventRelation
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Consequence_Management_Forum_Entry-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= Consequence_Management_Forum_Entry_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Consequence_Management_Forum_Entry__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Consequence_Management_Forum_Entry__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 19:14:16,826 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:14:16,826 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:14:16,829 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:14:16,830 INFO sqlalchemy.engine.Engine [generated in 0.00077s] ()
2026-09-03 19:14:16,833 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))


2026-09-03 19:14:16,834 INFO sqlalchemy.engine.Engine [generated in 0.00100s] ()
2026-09-03 19:14:16,836 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:14:16,836 INFO sqlalchemy.engine.Engine [generated in 0.00076s] ()
2026-09-03 19:14:16,838 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:14:16,839 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Consequence_Management_Forum_Entry-ALL]
;
2026-09-03 19:14:16,840 INFO sqlalchemy.engine.Engine [generated in 0.00108s] ()
2026-09-03 19:14:18,452 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 19:14:18,457 [INFO] Total Consequence_Management_Forum_Entry__c Records: 101
2026-09-03 19:14:18,458 [INFO] Unique in Metazoa_External_ID__c : 101
2026-09-03 19:14:18,458 [INFO] Preparing to upsert 101 records to Consequence_Management_Forum_Entry__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Client_s_impacted__c', 'Consequences__c', 'Description__c', 'EDP_External_Id__c', 'Explanation_to_support_Consequences__c', 'Meeting_date__c', 'Nature_of_client_detriment__c', 'Potential_dollar_loss_or_detriment__c', 'Received_Date__c', 'Recommendation__c', 'Root_Cause_s__c', 'Significant_Issue_Patterns__c', 'Source_of_Submission__c', 'Practice__c', 'Assurance_Review__c', 'Adviser__c', 'RecordTypeId']
2026-09-03 19:14:24,442 [INFO] Upsert complete. Time taken: 5.97s
2026-09-03 19:14:24,442 [INFO] Success: 101, Failures: 0
2026-09-03 19:14:24,447 [INFO] Load Results:      success  created                  id er

## 44 After Incident_File_Review__c - Assurance_Review__c - Run 2 - UPSERT, with Risk_Forum__c, Original_Request_Number__c, and Resubmission_case__c

In [53]:
"""
reference_to
["Account"]
["Assurance_Event__c"]
["Assurance_Review__c"]
["Assurance_Review__c"]
["Assurance_Review__c"]
["Consequence_Management_Forum_Entry__c"]
["Contact"]
["Contact"]
["Group", "User"]
["RecordType"]
["User"]
"""
"""
object_api_name
ASN_Indicator_History__c
Assurance_Review__c
Client_File__c
Consequence_Management_Forum_Entry__c
EmailMessage
Event
EventRelation
Incident_File_Review__c
QA__c
QA_Assignment_Hisotry__c
Question_Responses__c
Reviews_Accounted__c
SelectedQuestions_Targetedreviews__c
Task
TaskRelation
Vetting_Result__c
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Assurance_Review-ALL] where ([Original_Request_Number__c] > '' or [Resubmission_case__c] > '' or [Risk_Forum__c] > '')
;"""

from sf_target_fields import *
ls_tgt_objFields= ASSURANCE_REVIEW_TARGET_FIELDS
# ls_tgt_objFields = [x for x in ls_tgt_objFields if x not in ('Review__c')]
ls_tgt_objFields = ['Id', 'Risk_Forum__c', 'Original_Request_Number__c', 'Resubmission_case__c']

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

2026-09-03 19:15:56,555 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:15:56,556 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:15:56,559 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:15:56,560 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()


2026-09-03 19:15:56,562 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:15:56,563 INFO sqlalchemy.engine.Engine [generated in 0.00113s] ()
2026-09-03 19:15:56,565 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:15:56,565 INFO sqlalchemy.engine.Engine [generated in 0.00122s] ()
2026-09-03 19:15:56,565 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:15:56,565 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Assurance_Review-ALL] where ([Original_Request_Number__c] > '' or [Resubmission_case__c] > '' or [Risk_Forum__c] > '')
;
2026-09-03 19:15:56,572 INFO sqlalchemy.engine.Engine [generated in 0.00163s] ()
2026-09-03 19:16:05,289 INFO sqlalchemy.engine.Engine COMMIT


,Id,Risk_Forum__c,Original_Request_Number__c,Resubmission_case__c
0,a0oBm000004xYJcIAM,None,a0oBm000004xYJeIAM,None
1,a0oBm000004xYJdIAM,None,a0oBm000004xYJcIAM,None
2,a0oBm000004xYJfIAM,None,a0oBm000004xYJiIAM,None
3,a0oBm000004xYJgIAM,None,a0oBm000004xYJfIAM,None
4,a0oBm000004xYJhIAM,None,a0oBm000004xYJjIAM,None
...,...,...,...,...
2007,a0oBm000004xYklIAE,None,a0oBm000004xYmOIAU,None
2008,a0oBm000004xYVeIAM,None,a0oBm000004xYSaIAM,None
2009,a0oBm000004xYkyIAE,None,a0oBm000004xYmPIAU,None
2010,a0oBm000004xZQeIAM,None,a0oBm000004xZ7KIAU,None


In [54]:

logger.info(f"Total Assurance_Review__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Assurance_Review__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger,mode='UPSERT')
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Assurance_Review__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Case', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 19:16:18,070 [INFO] Total Assurance_Review__c Records: 2012
2026-09-03 19:16:18,085 [INFO] Preparing to upsert 2012 records to Assurance_Review__c with fields ['Id', 'Risk_Forum__c', 'Original_Request_Number__c', 'Resubmission_case__c']


2026-09-03 19:16:52,619 [INFO] Upsert complete. Time taken: 34.53s
2026-09-03 19:16:52,620 [INFO] Success: 2009, Failures: 3
2026-09-03 19:16:52,621 [WARNING] Failures: [{'success': False, 'created': False, 'id': 'a0oBm000004xYPLIA2', 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'Risk Forum: id value of incorrect type: a0D6F00000vjj07UAA+', 'fields': ['Risk_Forum__c']}]}, {'success': False, 'created': False, 'id': 'a0oBm000004xYP2IAM', 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'Risk Forum: id value of incorrect type: a0D6F00000uuXN6UAM+', 'fields': ['Risk_Forum__c']}]}, {'success': False, 'created': False, 'id': 'a0oBm000004xYP0IAM', 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'Risk Forum: id value of incorrect type: a0D6F00000utV0ZUAU+', 'fields': ['Risk_Forum__c']}]}]
2026-09-03 19:16:52,628 [INFO] Load Results:       success  created                  id errors
0        True    False  a0oBm000004xYJcIAM     []
1        True    False  a0oBm000004xYJdIAM    

## 50 After Assurance_Review__c - Client_File__c

In [55]:
"""
reference_to
["Assurance_Review__c"]
["User"]
"""
"""
object_api_name
EventRelation
Question_Responses__c
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Client_File-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= Client_File_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

2026-09-03 19:19:10,772 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:19:10,774 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:19:10,776 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:19:10,777 INFO sqlalchemy.engine.Engine [generated in 0.00091s] ()
2026-09-03 19:19:10,780 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:19:10,780 INFO sqlalchemy.engine.Engine [generated in 0.00081s] ()
2026-09-03 19:19:10,783 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:19:10,785 INFO sqlalchemy.engine.Engine [generated in 0.00167s] ()
2026-09-03 19:19:10,787 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:19:10,788 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Client_File-ALL]
;
2026-09-03 19:19:10,790 INFO sqlalchemy.engine.

,Metazoa_External_ID__c,CreatedById,CreatedDate,LastModifiedById,LastModifiedDate,Advice_Document_Type__c,Advice_Type__c,Application_Date__c,Chosen_for_Review__c,Client_Contact_Type__c,...,Preferred_Name__c,Products_Recommended__c,Products_recommended_Other__c,Products_Recommended_review__c,Reason__c,Reason_for_Choosing__c,Regulatory_Compliance_Score_Impact__c,Source_of_file__c,Total_Number_of_Issues__c,Review__c
0,a0B6F00001n0O5ZUAU,005W2000000K2d7IAC,2022-02-03T03:21:42.000+0000,005W2000000K2d7IAC,2022-02-03T04:37:20.000+0000,None,Direct Equities;Retirement Planning,None,Yes,None,...,Neville Lester,SPS Pension,None,None,"Occasional Advice (Execution Only, Time Critic...",None,0.0,State Operations Manager report,0.0,a0oBm000004xZANIA2
1,a0B6F00001n0S9SUAU,005W2000000K2d7IAC,2022-02-09T01:35:13.000+0000,005W2000000K2d7IAC,2022-02-09T01:35:13.000+0000,None,None,None,No,None,...,Gerda Renetzeder,None,None,None,Other file with similar characteristics chosen,None,None,State Operations Manager report,None,a0oBm000004xZALIA2
2,a0B6F00001n0S9cUAE,005W2000000K2d7IAC,2022-02-09T01:36:49.000+0000,005W2000000K2d7IAC,2022-04-13T02:56:23.000+0000,None,Direct Equities;Investments;Retirement Plannin...,2021-11-04,Yes,None,...,Simon & Beth Bamford S & B Bamford Super Fund,SMSF & Direct Equities,None,None,Review Advice,None,0.0,State Operations Manager report,1.0,a0oBm000004xZALIA2
3,a0B6F00001n0SApUAM,005W2000000K2d7IAC,2022-02-09T01:39:37.000+0000,005W2000000K2d7IAC,2022-04-13T02:56:47.000+0000,None,Direct Equities;Retirement Planning;Superannua...,2021-09-14,Yes,None,...,Kurt & Leanne Memish,BT Panorama Super & Pension,None,None,Normal Advice,None,0.0,State Operations Manager report,2.0,a0oBm000004xZALIA2
4,a0B6F00001n0SAqUAM,005W2000000K2d7IAC,2022-02-09T01:40:58.000+0000,005W2000000K2d7IAC,2022-04-13T02:52:14.000+0000,None,Direct Equities;Investments;Retirement Plannin...,2021-08-27,Yes,None,...,A & J McIntosh Super Fund John & Annette McIntosh,"SMSF, BT Panorama IP & Direct Equities",None,None,Specialised advice provided,None,0.0,State Operations Manager report,2.0,a0oBm000004xZALIA2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8597,a0BMq000008df57MAA,005Bm00000Y8KwIIAV,2026-02-23T07:45:07.000+0000,005Bm00000Y8KwIIAV,2026-02-23T07:45:07.000+0000,None,None,None,Yes,None,...,Shirley McDougall,None,None,None,Review Advice,Review Advice,None,Adviser Request,None,a0oBm000004xYrsIAE
8598,a0BMq000008d4qLMAQ,005Bm00000Y8KwIIAV,2026-02-23T00:38:30.000+0000,005Bm00000Y8KwIIAV,2026-02-23T00:38:30.000+0000,None,None,None,Yes,None,...,Stanley Wolfe,None,None,None,Review Advice,Review Advice,None,Adviser Request,None,a0oBm000004xYruIAE
8599,a0BMq000008TH5hMAG,005Bm00000Y8KwIIAV,2026-02-11T22:25:05.000+0000,005Bm00000Y8KwIIAV,2026-02-11T22:25:05.000+0000,None,None,None,Yes,None,...,Delma Boyd,None,None,None,Review Advice,Review Advice,None,Adviser Request,None,a0oBm000004xYrrIAE
8600,a0BMq000009bIdWMAU,005Bm00000Y8KwJIAV,2026-05-29T04:25:49.000+0000,005Bm00000Y8KwJIAV,2026-05-29T04:25:49.000+0000,None,None,None,Yes,None,...,Elizabeth and Sean Egan,None,None,None,Review Advice,Review Advice,None,Adviser Request,None,a0oBm000004xYI1IAM


In [57]:

logger.info(f"Total Client_File__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CaseComment', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Client_File__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=1000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 19:20:42,686 [INFO] Total Client_File__c Records: 8602
2026-09-03 19:20:42,693 [INFO] Unique in Metazoa_External_ID__c : 8602
2026-09-03 19:20:42,942 [INFO] Preparing to upsert 8602 records to Client_File__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Advice_Document_Type__c', 'Advice_Type__c', 'Application_Date__c', 'Chosen_for_Review__c', 'Client_Contact_Type__c', 'Client_File_Questionnaire_Status__c', 'ClientID_Xplan__c', 'Date_of_Advice__c', 'Date_Time_of_Phone_Call__c', 'EDP_External_Id__c', 'Fact_Find_Date_Signed__c', 'File_Selection_Date__c', 'General_Advice_Score_Impact__c', 'Number_of_General_Advice_Complaince__c', 'Number_of_High_Issues__c', 'Number_of_Low_Issues__c', 'Number_of_Medium_Issues__c', 'Number_of_Regulatory_Complaince__c', 'Preferred_Name__c', 'Products_Recommended__c', 'Products_recommended_Other__c', 'Products_Recommended_review__c', 'Reason__c', 'Reason_for_Choosing__c', 'Regulatory_Co

## 60 After Assurance_Review__c - QA__c

In [58]:
"""
reference_to
["Account"]
["Assurance_Review__c"]
["Contact"]
["Group", "User"]
["RecordType"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_QA-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= QA_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total QA__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='QA__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 19:23:59,996 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:23:59,996 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:23:59,999 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:24:00,000 INFO sqlalchemy.engine.Engine [generated in 0.00091s] ()
2026-09-03 19:24:00,003 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:24:00,004 INFO sqlalchemy.engine.Engine [generated in 0.00082s] ()
2026-09-03 19:24:00,006 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:24:00,006 INFO sqlalchemy.engine.Engine [generated in 0.00091s] ()
2026-09-03 19:24:00,009 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:24:00,010 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_QA-ALL]
;
2026-09-03 19:24:00,011 INFO sqlalchemy.engine.Engine [g

2026-09-03 19:24:00,347 [INFO] Total QA__c Records: 186
2026-09-03 19:24:00,349 [INFO] Unique in Metazoa_External_ID__c: 186
2026-09-03 19:24:00,360 [INFO] Preparing to upsert 186 records to QA__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Debrief__c', 'Debrief_Notes__c', 'Draft_Report__c', 'Final_Report__c', 'Interim_Report__c', 'Other_Discussion_Points__c', 'QA_Completed_On__c', 'QA_Date__c', 'QA_Outcome__c', 'QA_Status__c', 'QA_Type__c', 'Questionnaire_Launched_On__c', 'Questionnaire_Status__c', 'Review_Questionnaire__c', 'Scope_Strategies__c', 'Practice__c', 'Review__c', 'Adviser__c', 'RecordTypeId']
2026-09-03 19:24:06,490 [INFO] Upsert complete. Time taken: 6.13s
2026-09-03 19:24:06,491 [INFO] Success: 174, Failures: 12
2026-09-03 19:24:06,492 [WARNING] Failures: [{'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'Adviser: id value of incorrect type: 0

## 60 After Assurance_Review__c - QA_Assignment_History__c

In [59]:
"""
reference_to
["Assurance_Review__c"]
["Group", "User"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_QA_Assignment_Hisotry-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= QA_ASSIGNMENT_HISTORY_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total QA_Assignment_Hisotry__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='QA_Assignment_Hisotry__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 19:25:24,587 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:25:24,594 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:25:24,596 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:25:24,597 INFO sqlalchemy.engine.Engine [generated in 0.00086s] ()
2026-09-03 19:25:24,599 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:25:24,600 INFO sqlalchemy.engine.Engine [generated in 0.00083s] ()
2026-09-03 19:25:24,602 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:25:24,604 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ()
2026-09-03 19:25:24,606 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:25:24,607 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_QA_Assignment_Hisotry-ALL]
;
2026-09-03 19:25:24,608 INFO sqlalche

2026-09-03 19:25:24,822 [INFO] Total QA_Assignment_Hisotry__c Records: 395
2026-09-03 19:25:24,824 [INFO] Unique in Metazoa_External_ID__c: 395
2026-09-03 19:25:24,832 [INFO] Preparing to upsert 395 records to QA_Assignment_Hisotry__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Name', 'QA_Type__c', 'Assurance_Review__c', 'QA_Officer__c']
2026-09-03 19:25:31,043 [INFO] Upsert complete. Time taken: 6.21s
2026-09-03 19:25:31,045 [INFO] Success: 395, Failures: 0
2026-09-03 19:25:31,050 [INFO] Load Results:      success  created                  id errors
0       True     True  a1RBm00000QYxEnMAL     []
1       True     True  a1RBm00000QYxEoMAL     []
2       True     True  a1RBm00000QYxEpMAL     []
3       True     True  a1RBm00000QYxEqMAL     []
4       True     True  a1RBm00000QYxErMAL     []
..       ...      ...                 ...    ...
390     True     True  a1RBm00000QYxOJMA1     []
391     True     True  

## OBSOLETE - Paraplanning__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Paraplanning-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= PARAPLANNING_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Paraplanning__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=1000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Paraplanning__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## 70 After Assurance_Review__c - SelectedQuestions_Targetedreviews__c

In [60]:
"""
reference_to
["Assurance_Review__c"]
["Group", "User"]
["MasterQuestions_Targetedreviews__c"]
["User"]
"""
"""
object_api_name
EventRelation
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_SelectedQuestions_Targetedreviews-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= SelectedQuestions_Targetedreviews_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total SelectedQuestions_Targetedreviews__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='SelectedQuestions_Targetedreviews__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 19:26:39,412 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:26:39,413 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:26:39,416 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:26:39,417 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()
2026-09-03 19:26:39,420 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:26:39,420 INFO sqlalchemy.engine.Engine [generated in 0.00083s] ()
2026-09-03 19:26:39,424 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:26:39,425 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()


2026-09-03 19:26:39,427 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:26:39,427 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_SelectedQuestions_Targetedreviews-ALL]
;
2026-09-03 19:26:39,427 INFO sqlalchemy.engine.Engine [generated in 0.00145s] ()
2026-09-03 19:26:39,713 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 19:26:39,717 [INFO] Total SelectedQuestions_Targetedreviews__c Records: 190
2026-09-03 19:26:39,719 [INFO] Unique in Metazoa_External_ID__c : 190
2026-09-03 19:26:39,723 [INFO] Preparing to upsert 190 records to SelectedQuestions_Targetedreviews__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Cloned__c', 'Question_Number__c', 'Question_Text__c', 'Sorting_Index__c', 'Assurance_Review__c', 'MasterQuestions_Targetedreviews__c']
2026-09-03 19:26:40,589 [INFO] Upsert complete. Time taken: 0.87s
2026-09-03 19:26:40,589 [INFO] Success: 82, Failures: 108
2026-09-03 19:26:40,594 [WARNING] Failures: [{'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'Assurance Review: id value of incorrect type: a09Mq00000358MbIAI+', 'fields': ['Assurance_Review__c']}]}, {'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'Ass

## 70 After Assurance_Review__c - Question_Responses__c

In [6]:
"""
reference_to
["Account"]
["Assurance_Review__c"]
["Client_File__c"]
["Contact"]
["Group", "User"]
["Incident_File_Review__c"]
["Master_Questions__c"]
["MasterQuestions_Targetedreviews__c"]
["PSA__c"]
["QA__c"]
["RecordType"]
["User"]
["Vetting_Result__c"]
"""
"""
object_api_name
EmailMessage
Event
Event
EventRelation
Response_Finding__c
Snapshot_Data__c
Task
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select top 50000 * from [SalesforceMigration].[sfrhd].[vw_upsert_Question_Responses-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= QUESTION_RESPONSES_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Question_Responses__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=500, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 19:30:45,398 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 19:30:45,399 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 19:30:45,402 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 19:30:45,403 INFO sqlalchemy.engine.Engine [generated in 0.00091s] ()
2026-09-03 19:30:45,405 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 19:30:45,405 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()
2026-09-03 19:30:45,405 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 19:30:45,405 INFO sqlalchemy.engine.Engine [generated in 0.00068s] ()
2026-09-03 19:30:45,405 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 19:30:45,405 INFO sqlalchemy.engine.Engine select top 50000 * from [SalesforceMigration].[sfrhd].[vw_upsert_Question_Responses-ALL]
;
2026-09-03 19:30:45,405 INFO s

2026-09-03 19:31:30,083 [INFO] Total Question_Responses__c Records: 50000
2026-09-03 19:31:30,103 [INFO] Unique in Metazoa_External_ID__c: 50000
2026-09-03 19:31:32,155 [INFO] Preparing to upsert 50000 records to Question_Responses__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Actions_Taken__c', 'Actual_Loss_in_terms__c', 'Advice_Type__c', 'Comment__c', 'Comment_Yes__c', 'Comments__c', 'Date_Paid__c', 'Document_Name__c', 'EDP_External_Id__c', 'External_ID__c', 'Feedback__c', 'Has_Action__c', 'Incident_Close_Date__c', 'Incident_Status__c', 'Key_Area__c', 'Licensee__c', 'Mark_as_Confidential__c', 'Partner_AFSL__c', 'Pre_Review_and_Overview_Response__c', 'Prior_Owner__c', 'QA_Rationale__c', 'QA_Response__c', 'Question_Number__c', 'Question_Text__c', 'Rationale__c', 'Report_Breach__c', 'Requires_Action__c', 'Response__c', 'Responsibility__c', 'Root_Cause__c', 'Practice__c', 'Assurance_Review__c', 'Client_File__c'

## 20 After User - A1_Support_Staff_Registrations__c

In [7]:
"""
reference_to
["Contact"]
["Group", "User"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_A1_Support_Staff_Registrations-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= A1_Support_Staff_Registrations_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total A1_Support_Staff_Registrations__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Id'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='A1_Support_Staff_Registrations__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:14:49,748 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:14:49,749 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 20:14:49,752 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 20:14:49,753 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ()


2026-09-03 20:14:49,755 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:14:49,756 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ()
2026-09-03 20:14:49,759 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:14:49,759 INFO sqlalchemy.engine.Engine [generated in 0.00103s] ()
2026-09-03 20:14:49,762 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:14:49,763 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_A1_Support_Staff_Registrations-ALL]
;
2026-09-03 20:14:49,764 INFO sqlalchemy.engine.Engine [generated in 0.00145s] ()
2026-09-03 20:14:49,886 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 20:14:49,932 [INFO] Total A1_Support_Staff_Registrations__c Records: 5
2026-09-03 20:14:49,937 [INFO] Preparing to upsert 5 records to A1_Support_Staff_Registrations__c with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Approval_Status__c', 'Email_Address__c', 'First_Name__c', 'Last_Name__c', 'Role__c', 'Adviser__c']
2026-09-03 20:14:50,868 [INFO] Upsert complete. Time taken: 0.93s
2026-09-03 20:14:50,869 [INFO] Success: 5, Failures: 0
2026-09-03 20:14:50,874 [INFO] Load Results:    success  created                  id errors
0     True     True  a0QBm00000T0M34MAF     []
1     True     True  a0QBm00000T0M35MAF     []
2     True     True  a0QBm00000T0M36MAF     []
3     True     True  a0QBm00000T0M37MAF     []
4     True     True  a0QBm00000T0M38MAF     []


## Phase 2 - Product_Questions__c

In [8]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Product_Questions-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= PRODUCT_QUESTIONS_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Product_Questions__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Product_Questions__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:15:25,666 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:15:25,667 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 20:15:25,669 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 20:15:25,670 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()
2026-09-03 20:15:25,673 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:15:25,674 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()
2026-09-03 20:15:25,676 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:15:25,678 INFO sqlalchemy.engine.Engine [generated in 0.00099s] ()
2026-09-03 20:15:25,679 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:15:25,680 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Product_Questions-ALL]
;
2026-09-03 20:15:25,682 INFO sqlalchemy.e

2026-09-03 20:15:26,615 [INFO] Total Product_Questions__c Records: 425
2026-09-03 20:15:26,616 [INFO] Unique in Id: 425
2026-09-03 20:15:26,623 [INFO] Preparing to upsert 425 records to Product_Questions__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'EDP_External_Id__c', 'ExternalID__c', 'Master_Question__c', 'Product__c']
2026-09-03 20:15:32,633 [INFO] Upsert complete. Time taken: 6.01s
2026-09-03 20:15:32,634 [INFO] Success: 425, Failures: 0
2026-09-03 20:15:32,639 [INFO] Load Results:      success  created                  id errors
0       True     True  a1QBm000002yq7RMAQ     []
1       True     True  a1QBm000002yq7SMAQ     []
2       True     True  a1QBm000002yq7TMAQ     []
3       True     True  a1QBm000002yq7UMAQ     []
4       True     True  a1QBm000002yq7VMAQ     []
..       ...      ...                 ...    ...
420     True     True  a1QBm000002yqEDMAY     []
421     True     True  a1QBm000002yqEE

## Phase 2 - Master_workshop__c

In [9]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_workshop-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= MASTER_WORKSHOP_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Master_workshop__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Master_workshop__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:16:06,631 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:16:06,632 INFO sqlalchemy.engine.Engine [raw sql] ()


2026-09-03 20:16:06,635 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 20:16:06,636 INFO sqlalchemy.engine.Engine [generated in 0.00091s] ()
2026-09-03 20:16:06,636 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:16:06,636 INFO sqlalchemy.engine.Engine [generated in 0.00087s] ()
2026-09-03 20:16:06,636 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:16:06,636 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ()
2026-09-03 20:16:06,644 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:16:06,646 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_workshop-ALL]
;
2026-09-03 20:16:06,647 INFO sqlalchemy.engine.Engine [generated in 0.00127s] ()
2026-09-03 20:16:06,697 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 20:16:06,702 [INFO] Total Master_workshop__c Records: 4
2026-09-03 20:16:06,704 [INFO] Preparing to upsert 4 records to Master_workshop__c with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'IsActive__c', 'Name', 'RecordTypeId']
2026-09-03 20:16:07,735 [INFO] Upsert complete. Time taken: 1.03s
2026-09-03 20:16:07,736 [INFO] Success: 4, Failures: 0
2026-09-03 20:16:07,739 [INFO] Load Results:    success  created                  id errors
0     True     True  a1JBm00000RzTLhMAN     []
1     True     True  a1JBm00000RzTLiMAN     []
2     True     True  a1JBm00000RzTLjMAN     []
3     True     True  a1JBm00000RzTLkMAN     []


## Phase 2 - Master_Module__c

In [10]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Module-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= MASTER_MODULE_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Master_Module__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Master_Module__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:16:41,786 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:16:41,787 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 20:16:41,790 INFO sqlalchemy.engine.Engine SELECT schema_name()


2026-09-03 20:16:41,790 INFO sqlalchemy.engine.Engine [generated in 0.00098s] ()
2026-09-03 20:16:41,793 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:16:41,794 INFO sqlalchemy.engine.Engine [generated in 0.00090s] ()
2026-09-03 20:16:41,796 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:16:41,797 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()
2026-09-03 20:16:41,800 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:16:41,801 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Module-ALL]
;
2026-09-03 20:16:41,801 INFO sqlalchemy.engine.Engine [generated in 0.00083s] ()
2026-09-03 20:16:41,854 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 20:16:41,856 [INFO] Total Master_Module__c Records: 6
2026-09-03 20:16:41,858 [INFO] Preparing to upsert 6 records to Master_Module__c with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'IsActive__c', 'Name', 'Master_workshop__c', 'RecordTypeId']
2026-09-03 20:16:42,704 [INFO] Upsert complete. Time taken: 0.85s
2026-09-03 20:16:42,705 [INFO] Success: 6, Failures: 0
2026-09-03 20:16:42,709 [INFO] Load Results:    success  created                  id errors
0     True     True  a1IBm000005hPLFMA2     []
1     True     True  a1IBm000005hPLGMA2     []
2     True     True  a1IBm000005hPLHMA2     []
3     True     True  a1IBm000005hPLIMA2     []
4     True     True  a1IBm000005hPLJMA2     []
5     True     True  a1IBm000005hPLKMA2     []


## Phase 2 - Master_Topic__c

In [11]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Topic-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= MASTER_TOPIC_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Master_Topic__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Master_Topic__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:17:13,098 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:17:13,099 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 20:17:13,101 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 20:17:13,102 INFO sqlalchemy.engine.Engine [generated in 0.00114s] ()
2026-09-03 20:17:13,104 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:17:13,105 INFO sqlalchemy.engine.Engine [generated in 0.00097s] ()
2026-09-03 20:17:13,107 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:17:13,108 INFO sqlalchemy.engine.Engine [generated in 0.00098s] ()
2026-09-03 20:17:13,110 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:17:13,111 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Topic-ALL]
;
2026-09-03 20:17:13,112 INFO sqlalchemy.engine

2026-09-03 20:17:13,198 [INFO] Total Master_Topic__c Records: 23
2026-09-03 20:17:13,202 [INFO] Preparing to upsert 23 records to Master_Topic__c with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'IsActive__c', 'Name', 'Order__c', 'Master_module__c', 'RecordTypeId']
2026-09-03 20:17:14,162 [INFO] Upsert complete. Time taken: 0.96s
2026-09-03 20:17:14,164 [INFO] Success: 23, Failures: 0
2026-09-03 20:17:14,170 [INFO] Load Results:     success  created                  id errors
0      True     True  a1HBm000004xf3FMAQ     []
1      True     True  a1HBm000004xf3GMAQ     []
2      True     True  a1HBm000004xf3HMAQ     []
3      True     True  a1HBm000004xf3IMAQ     []
4      True     True  a1HBm000004xf3JMAQ     []
5      True     True  a1HBm000004xf3KMAQ     []
6      True     True  a1HBm000004xf3LMAQ     []
7      True     True  a1HBm000004xf3MMAQ     []
8      True     True  a1HBm000004xf3NMAQ     []
9      True     True  a1HBm000004xf3OMAQ  

## Phase 2 - Master_Item__c

In [12]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Item-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= MASTER_ITEM_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Master_Item__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Master_Item__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:17:45,410 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:17:45,411 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 20:17:45,414 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 20:17:45,415 INFO sqlalchemy.engine.Engine [generated in 0.00124s] ()


2026-09-03 20:17:45,417 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:17:45,418 INFO sqlalchemy.engine.Engine [generated in 0.00134s] ()
2026-09-03 20:17:45,418 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:17:45,418 INFO sqlalchemy.engine.Engine [generated in 0.00073s] ()
2026-09-03 20:17:45,418 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:17:45,426 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Master_Item-ALL]
;
2026-09-03 20:17:45,427 INFO sqlalchemy.engine.Engine [generated in 0.00126s] ()
2026-09-03 20:17:45,521 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 20:17:45,525 [INFO] Total Master_Item__c Records: 123
2026-09-03 20:17:45,526 [INFO] Unique in Id: 123
2026-09-03 20:17:45,531 [INFO] Preparing to upsert 123 records to Master_Item__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Description__c', 'IsActive__c', 'Mandatory__c', 'Name', 'Master_Topic__c', 'RecordTypeId']
2026-09-03 20:17:46,429 [INFO] Upsert complete. Time taken: 0.90s
2026-09-03 20:17:46,430 [INFO] Success: 123, Failures: 0
2026-09-03 20:17:46,434 [INFO] Load Results:      success  created                  id errors
0       True     True  a1FBm000002XUXdMAO     []
1       True     True  a1FBm000002XUXeMAO     []
2       True     True  a1FBm000002XUXfMAO     []
3       True     True  a1FBm000002XUXgMAO     []
4       True     True  a1FBm000002XUXhMAO     []
..       ...      ...                 ...    ...
118     True     True  a1FBm000002XUZXMA4     []
119     True     True  a1FBm00000

## Phase 2 - Workshop__c

In [13]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Workshop-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= WORKSHOP_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Workshop__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Workshop__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:18:16,593 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:18:16,594 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 20:18:16,597 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 20:18:16,598 INFO sqlalchemy.engine.Engine [generated in 0.00102s] ()
2026-09-03 20:18:16,601 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:18:16,601 INFO sqlalchemy.engine.Engine [generated in 0.00085s] ()
2026-09-03 20:18:16,604 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:18:16,605 INFO sqlalchemy.engine.Engine [generated in 0.00106s] ()
2026-09-03 20:18:16,607 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:18:16,608 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Workshop-ALL]
;
2026-09-03 20:18:16,609 INFO sqlalchemy.engine.Eng

2026-09-03 20:18:16,659 [INFO] Total Workshop__c Records: 14
2026-09-03 20:18:16,660 [INFO] Unique in Id: 14
2026-09-03 20:18:16,663 [INFO] Preparing to upsert 14 records to Workshop__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Type_of_Workshop__c', 'UniqueMasterWorkshopAA__c', 'Workshop_Attended_Date__c', 'Adviser_Academy__c', 'Workshop__c', 'RecordTypeId']
2026-09-03 20:18:22,410 [INFO] Upsert complete. Time taken: 5.75s
2026-09-03 20:18:22,411 [INFO] Success: 14, Failures: 0
2026-09-03 20:18:22,416 [INFO] Load Results:     success  created                  id errors
0      True     True  a1vBm000001m6aPIAQ     []
1      True     True  a1vBm000001m6aQIAQ     []
2      True     True  a1vBm000001m6aRIAQ     []
3      True     True  a1vBm000001m6aSIAQ     []
4      True     True  a1vBm000001m6aTIAQ     []
5      True     True  a1vBm000001m6aUIAQ     []
6      True     True  a1vBm000001m6aVIAQ     []
7      True     True

## Phase 2 - Module__c

In [14]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Module-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= MODULE_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Module__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Module__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:18:49,931 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:18:49,933 INFO sqlalchemy.engine.Engine [raw sql] ()


2026-09-03 20:18:49,935 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 20:18:49,936 INFO sqlalchemy.engine.Engine [generated in 0.00115s] ()
2026-09-03 20:18:49,936 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:18:49,936 INFO sqlalchemy.engine.Engine [generated in 0.00094s] ()
2026-09-03 20:18:49,936 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:18:49,936 INFO sqlalchemy.engine.Engine [generated in 0.00086s] ()
2026-09-03 20:18:49,944 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:18:49,945 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Module-ALL]
;
2026-09-03 20:18:49,947 INFO sqlalchemy.engine.Engine [generated in 0.00201s] ()
2026-09-03 20:18:50,005 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 20:18:50,009 [INFO] Total Module__c Records: 21
2026-09-03 20:18:50,010 [INFO] Unique in Id: 21
2026-09-03 20:18:50,014 [INFO] Preparing to upsert 21 records to Module__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Actual_Complete_Date__c', 'Actual_Start_Date__c', 'Comments__c', 'Completion__c', 'Description__c', 'Implementing__c', 'Planned_Complete_Date__c', 'Planned_Start_Date__c', 'Responsibility__c', 'Total_Completion_Topics__c', 'Total_Topics__c', 'Master_module__c', 'RecordTypeId', 'Workshop__c']
2026-09-03 20:18:50,922 [INFO] Upsert complete. Time taken: 0.91s
2026-09-03 20:18:50,923 [INFO] Success: 21, Failures: 0
2026-09-03 20:18:50,930 [INFO] Load Results:     success  created                  id errors
0      True     True  a1KBm00000IspETMAZ     []
1      True     True  a1KBm00000IspEUMAZ     []
2      True     True  a1KBm00000IspEVMAZ     []
3      True     True  a1KBm00000IspEWMAZ     []
4      T

## Phase 2 - Topic__c

In [15]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Topic-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= TOPIC_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Topic__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Topic__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:19:24,686 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:19:24,687 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 20:19:24,689 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 20:19:24,690 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()
2026-09-03 20:19:24,694 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:19:24,695 INFO sqlalchemy.engine.Engine [generated in 0.00142s] ()
2026-09-03 20:19:24,698 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:19:24,699 INFO sqlalchemy.engine.Engine [generated in 0.00089s] ()
2026-09-03 20:19:24,700 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:19:24,702 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Topic-ALL]
;
2026-09-03 20:19:24,703 INFO sqlalchemy.engine.Engine

2026-09-03 20:19:24,829 [INFO] Total Topic__c Records: 609
2026-09-03 20:19:24,831 [INFO] Unique in Id: 609
2026-09-03 20:19:24,846 [INFO] Preparing to upsert 609 records to Topic__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Actual_Complete_Date__c', 'Actual_Start_Date__c', 'Comments__c', 'Planned_Complete_Date__c', 'Planned_Start_Date__c', 'Responsibility__c', 'Master_Topic__c', 'Module__c', 'RecordTypeId']
2026-09-03 20:19:31,038 [INFO] Upsert complete. Time taken: 6.19s
2026-09-03 20:19:31,039 [INFO] Success: 609, Failures: 0
2026-09-03 20:19:31,045 [INFO] Load Results:      success  created                  id errors
0       True     True  a1mBm000003OxRpIAK     []
1       True     True  a1mBm000003OxRqIAK     []
2       True     True  a1mBm000003OxRrIAK     []
3       True     True  a1mBm000003OxRsIAK     []
4       True     True  a1mBm000003OxRtIAK     []
..       ...      ...                 ...    ..

## Phase 2 - Email_Log__c - RUN 1 - INSERT without CSA_Fee_Switch_Off_History__c

In [16]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Email_Log-Run1]
;"""

from sf_target_fields import *

ls_tgt_objFields= Email_Log_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Email_Log__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Email_Log__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:20:37,724 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:20:37,725 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 20:20:37,728 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 20:20:37,729 INFO sqlalchemy.engine.Engine [generated in 0.00085s] ()
2026-09-03 20:20:37,731 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:20:37,732 INFO sqlalchemy.engine.Engine [generated in 0.00076s] ()


2026-09-03 20:20:37,734 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:20:37,735 INFO sqlalchemy.engine.Engine [generated in 0.00086s] ()
2026-09-03 20:20:37,738 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:20:37,739 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Email_Log-Run1]
;
2026-09-03 20:20:37,741 INFO sqlalchemy.engine.Engine [generated in 0.00202s] ()
2026-09-03 20:20:38,305 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 20:20:38,311 [INFO] Total Email_Log__c Records: 1322
2026-09-03 20:20:38,311 [INFO] Unique in Metazoa_External_ID__c: 1322
2026-09-03 20:20:38,350 [INFO] Preparing to upsert 1322 records to Email_Log__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Email__c', 'Email_Sent_Date__c', 'Email1__c', 'Email2__c', 'First_Notification_Sent__c', 'First_Notification_Sent_Date__c', 'Second_Notification_Sent__c', 'Second_Notification_Sent_Date__c', 'Adviser__c', 'Fee_Switch_Off_History__c', 'Knowledge__c', 'RecordTypeId', 'Third_Party_Provider__c']
2026-09-03 20:20:44,342 [INFO] Upsert complete. Time taken: 5.99s
2026-09-03 20:20:44,343 [INFO] Success: 1322, Failures: 0
2026-09-03 20:20:44,349 [INFO] Load Results:       success  created                  id errors
0        True     True  a17Bm000090w91cIAA     []
1        True     True  a17Bm000090w91dIAA     []
2        True     True  a17Bm000090w91eIAA     []
3  

## Phase 2 - CSA_Fee_Switch_Off_History__c - Must run AFTER Email_Log__c Run 1 - These records has Email_Log__c and is SAFE to run UPSERT if needed.
### 1,305 Records targeted

In [17]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CSA_Fee_Switch_Off_History-Email]
;"""

from sf_target_fields import *

ls_tgt_objFields= CSA_Fee_Switch_Off_History_TARGET_FIELDS
ls_tgt_objFields = [x for x in ls_tgt_objFields if x not in ('Name')]

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total CSA_Fee_Switch_Off_History__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Id'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CSA_Fee_Switch_Off_History__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 20:21:30,878 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 20:21:30,879 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 20:21:30,881 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 20:21:30,883 INFO sqlalchemy.engine.Engine [generated in 0.00097s] ()
2026-09-03 20:21:30,885 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 20:21:30,886 INFO sqlalchemy.engine.Engine [generated in 0.00074s] ()
2026-09-03 20:21:30,888 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 20:21:30,889 INFO sqlalchemy.engine.Engine [generated in 0.00091s] ()


2026-09-03 20:21:30,891 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 20:21:30,892 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_CSA_Fee_Switch_Off_History-Email]
;
2026-09-03 20:21:30,893 INFO sqlalchemy.engine.Engine [generated in 0.00109s] ()
2026-09-03 20:21:30,898 INFO sqlalchemy.engine.Engine ROLLBACK


ProgrammingError: (pyodbc.ProgrammingError) ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'SalesforceMigration.sf.ASF_CSA_Case__c'. (208) (SQLExecDirectW); [42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Could not use view or function 'SalesforceMigration.sfrhd.vw_upsert_CSA_Fee_Switch_Off_History-Email' because of binding errors. (4413)")
[SQL: select * from [SalesforceMigration].[sfrhd].[vw_upsert_CSA_Fee_Switch_Off_History-Email]
;]
(Background on this error at: https://sqlalche.me/e/20/f405)

## Phase 2 - CSA_Fee_Switch_Off_History__c - Must run AFTER Email_Log__c Run 1 - These records do NOT have Email_Log__c and is NOT safe to run UPSERT. Only INSERT once.
### 1,732 Records targeted

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CSA_Fee_Switch_Off_History-NoEmail]
;"""

from sf_target_fields import *

ls_tgt_objFields= CSA_Fee_Switch_Off_History_TARGET_FIELDS
ls_tgt_objFields = [x for x in ls_tgt_objFields if x not in ('Name')]

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total CSA_Fee_Switch_Off_History__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Id'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CSA_Fee_Switch_Off_History__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - Email_Log__c - RUN 2 - UPSERT on CSA_Fee_Switch_Off_History__c

In [ ]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Email_Log-Run2Upsert]
;"""

from sf_target_fields import *

ls_tgt_objFields= [
    'Id'
    ,'Metazoa_External_ID__c'
    ,'Fee_Switch_Off_History__c'
]

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Email_Log__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Email_Log__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger,mode='UPSERT')
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Email_Log__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger,mode='UPSERT')

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## Phase 2 - AdviserOne_Event__c

In [5]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_AdviserOne_Event-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= AdviserOne_Event_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total AdviserOne_Event__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Id: {df_transformed['Id'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AdviserOne_Event__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='AccountContactRelation', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 21:57:11,056 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 21:57:11,057 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 21:57:11,060 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 21:57:11,061 INFO sqlalchemy.engine.Engine [generated in 0.00091s] ()
2026-09-03 21:57:11,061 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 21:57:11,061 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()
2026-09-03 21:57:11,067 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 21:57:11,067 INFO sqlalchemy.engine.Engine [generated in 0.00079s] ()
2026-09-03 21:57:11,070 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 21:57:11,070 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_AdviserOne_Event-ALL]
;
2026-09-03 21:57:11,071 INFO sqlalchemy.en

2026-09-03 21:57:11,158 [INFO] Total AdviserOne_Event__c Records: 28
2026-09-03 21:57:11,165 [INFO] Preparing to upsert 28 records to AdviserOne_Event__c with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Article__c', 'Detail_URL__c', 'Display_on_Home_Page__c', 'Display_URL__c', 'Event_Date__c', 'Event_Registration_Link__c', 'Event_Summary__c', 'Event_Time__c', 'Event_Type__c', 'Home_Page_Display_Expiry__c', 'Licensees__c', 'Licensees_Formula__c', 'Provider__c']
2026-09-03 21:57:12,336 [INFO] Upsert complete. Time taken: 1.17s
2026-09-03 21:57:12,338 [INFO] Success: 28, Failures: 0
2026-09-03 21:57:12,362 [INFO] Load Results:     success  created                  id errors
0      True     True  a0dBm00000D0rBJIAZ     []
1      True     True  a0dBm00000D0rBKIAZ     []
2      True     True  a0dBm00000D0rBLIAZ     []
3      True     True  a0dBm00000D0rBMIAZ     []
4      True     True  a0dBm00000D0rBNIAZ     []
5      True     True  a0dBm00000D0

## Phase 2 - Approval__c

In [6]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Approval-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= Approval_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Approval__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Approval__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Approval__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 21:59:26,340 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 21:59:26,341 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 21:59:26,344 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 21:59:26,345 INFO sqlalchemy.engine.Engine [generated in 0.00102s] ()
2026-09-03 21:59:26,348 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 21:59:26,349 INFO sqlalchemy.engine.Engine [generated in 0.00090s] ()
2026-09-03 21:59:26,350 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 21:59:26,350 INFO sqlalchemy.engine.Engine [generated in 0.00089s] ()
2026-09-03 21:59:26,350 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 21:59:26,350 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Approval-ALL]
;
2026-09-03 21:59:26,350 INFO sqlalchemy.engine.Eng

2026-09-03 21:59:26,454 [INFO] Total Approval__c Records: 442
2026-09-03 21:59:26,459 [INFO] Unique in Metazoa_External_ID__c : 442
2026-09-03 21:59:26,468 [INFO] Preparing to upsert 442 records to Approval__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Approval_Decision__c', 'Comments__c', 'Decision_Date__c', 'EDP_External_Id__c', 'Name', 'Onboarding_Offboarding_Application__c', 'Approver__c']
2026-09-03 21:59:32,497 [INFO] Upsert complete. Time taken: 6.03s
2026-09-03 21:59:32,498 [INFO] Success: 442, Failures: 0
2026-09-03 21:59:32,505 [INFO] Load Results:      success  created                  id errors
0       True     True  a0jBm000007uGSfIAM     []
1       True     True  a0jBm000007uGSgIAM     []
2       True     True  a0jBm000007uGShIAM     []
3       True     True  a0jBm000007uGSiIAM     []
4       True     True  a0jBm000007uGSjIAM     []
..       ...      ...                 ...    ...
437     True     True  a0

## Phase 2 - Area_of_Advice__c

In [7]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Area_of_Advice-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= Area_of_Advice_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Area_of_Advice__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Approval__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Area_of_Advice__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:00:02,697 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:00:02,697 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:00:02,697 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:00:02,697 INFO sqlalchemy.engine.Engine [generated in 0.00090s] ()
2026-09-03 22:00:02,697 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:00:02,697 INFO sqlalchemy.engine.Engine [generated in 0.00115s] ()
2026-09-03 22:00:02,706 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:00:02,707 INFO sqlalchemy.engine.Engine [generated in 0.00086s] ()
2026-09-03 22:00:02,709 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:00:02,711 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Area_of_Advice-ALL]
;
2026-09-03 22:00:02,711 INFO sqlalchemy.engi

2026-09-03 22:00:02,852 [INFO] Total Area_of_Advice__c Records: 825
2026-09-03 22:00:02,853 [INFO] Unique in Metazoa_External_ID__c : 825
2026-09-03 22:00:02,870 [INFO] Preparing to upsert 825 records to Area_of_Advice__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Current_File_Number_for_Phase__c', 'Current_Vetting_Phase__c', 'Document_Received_Date__c', 'EDP_External_Id__c', 'Next_Vetting_Phase__c', 'Number_of_Files_Approved__c', 'Number_of_Files_in_Phase_Approved__c', 'Number_of_Files_in_Phase_Not_Approved__c', 'Number_of_Files_Not_Approved__c', 'Number_of_Files_Submitted__c', 'Reason_for_Vetting__c', 'Vetting_Cleared_Date__c', 'Vetting_Start_Date__c', 'Vetting_Status__c', 'Adviser__c', 'Area_of_Advice__c']
2026-09-03 22:00:14,136 [INFO] Upsert complete. Time taken: 11.26s
2026-09-03 22:00:14,137 [INFO] Success: 825, Failures: 0
2026-09-03 22:00:14,143 [INFO] Load Results:      success  created                  id err

## Phase 2 - Breach_Reporting__c

In [8]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Breach_Reporting-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= Breach_Reporting_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Breach_Reporting__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Approval__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Breach_Reporting__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:00:43,961 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:00:43,962 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:00:43,965 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:00:43,966 INFO sqlalchemy.engine.Engine [generated in 0.00109s] ()
2026-09-03 22:00:43,968 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:00:43,969 INFO sqlalchemy.engine.Engine [generated in 0.00099s] ()
2026-09-03 22:00:43,971 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:00:43,972 INFO sqlalchemy.engine.Engine [generated in 0.00102s] ()
2026-09-03 22:00:43,974 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:00:43,976 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Breach_Reporting-ALL]
;
2026-09-03 22:00:43,977 INFO sqlalchemy.en

2026-09-03 22:00:46,664 [INFO] Total Breach_Reporting__c Records: 540
2026-09-03 22:00:46,665 [INFO] Unique in Metazoa_External_ID__c : 540
2026-09-03 22:00:46,707 [INFO] Preparing to upsert 540 records to Breach_Reporting__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'A10__c', 'A2__c', 'A3__c', 'A4__c', 'A5__c', 'A6__c', 'A8__c', 'ASIC_Breach_Reference__c', 'ASIC_Lodgement_status__c', 'ASIC_Update_Date__c', 'BRC_Date__c', 'BRC_For_Noting_Date__c', 'Breach_Details__c', 'Breach_Report_date__c', 'Breach_Status__c', 'Client_Loss_Investigation_Complete_Date__c', 'Client_Loss_Investigation_Start_date__c', 'Client_Loss_Remediation_Completed_Date__c', 'Client_Notice_Period_start_date__c', 'Client_Notification_sent_date__c', 'Date_Of_The_First_Instance_Of_The_Breach__c', 'Date_Of_The_Last_Instance_Of_The_Breach__c', 'DNU_Breach_report_required_date__c', 'Estimated_Completion_Of_Investigation__c', 'External_AFSL__c', 'Incomplete_

## Phase 2 - CaseComment

In [9]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CaseComment-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= CaseComment_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total CaseComment Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CaseComment', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Breach_Reporting__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:01:34,609 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:01:34,610 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:01:34,613 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:01:34,614 INFO sqlalchemy.engine.Engine [generated in 0.00089s] ()
2026-09-03 22:01:34,616 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:01:34,618 INFO sqlalchemy.engine.Engine [generated in 0.00087s] ()
2026-09-03 22:01:34,620 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:01:34,621 INFO sqlalchemy.engine.Engine [generated in 0.00096s] ()
2026-09-03 22:01:34,622 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:01:34,623 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_CaseComment-ALL]
;
2026-09-03 22:01:34,624 INFO sqlalchemy.engine.

2026-09-03 22:01:34,861 [INFO] Total CaseComment Records: 12
2026-09-03 22:01:34,863 [INFO] Preparing to upsert 12 records to CaseComment with fields ['CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'CommentBody', 'IsPublished', 'ParentId']
2026-09-03 22:01:42,057 [INFO] Upsert complete. Time taken: 7.19s
2026-09-03 22:01:42,058 [INFO] Success: 12, Failures: 0
2026-09-03 22:01:42,063 [INFO] Load Results:     success  created                  id errors
0      True     True  00aBm00000OHBneIAH     []
1      True     True  00aBm00000OHBnfIAH     []
2      True     True  00aBm00000OHBngIAH     []
3      True     True  00aBm00000OHBnhIAH     []
4      True     True  00aBm00000OHBniIAH     []
5      True     True  00aBm00000OHBnjIAH     []
6      True     True  00aBm00000OHBnkIAH     []
7      True     True  00aBm00000OHBnlIAH     []
8      True     True  00aBm00000OHBnmIAH     []
9      True     True  00aBm00000OHBnnIAH     []
10     True     True  00aBm00000OHBnoIAH  

## Phase 2 - InitiativeChecklist__c

In [10]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_InitiativeChecklist-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= INITIATIVECHECKLIST_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total InitiativeChecklist__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='InitiativeChecklist__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:02:14,666 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:02:14,666 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:02:14,670 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:02:14,671 INFO sqlalchemy.engine.Engine [generated in 0.00087s] ()
2026-09-03 22:02:14,674 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:02:14,675 INFO sqlalchemy.engine.Engine [generated in 0.00081s] ()


2026-09-03 22:02:14,677 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:02:14,679 INFO sqlalchemy.engine.Engine [generated in 0.00238s] ()
2026-09-03 22:02:14,683 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:02:14,685 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_InitiativeChecklist-ALL]
;
2026-09-03 22:02:14,686 INFO sqlalchemy.engine.Engine [generated in 0.00144s] ()
2026-09-03 22:02:14,926 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 22:02:14,931 [INFO] Total InitiativeChecklist__c Records: 2300
2026-09-03 22:02:14,933 [INFO] Unique in Metazoa_External_ID__c : 2300
2026-09-03 22:02:14,972 [INFO] Preparing to upsert 2300 records to InitiativeChecklist__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Action_Required__c', 'Assessment_Type__c', 'Close_Date__c', 'CyberSecurity_Remediation_Comment__c', 'Due_Date__c', 'Issue_Identified__c', 'Practice_ID__c', 'Priority__c', 'Start_Date__c', 'Status__c', 'Initiative_Implementation__c', 'SecurityMeasure__c', 'RecordTypeId']
2026-09-03 22:02:31,408 [INFO] Upsert complete. Time taken: 16.44s
2026-09-03 22:02:31,409 [INFO] Success: 2275, Failures: 25
2026-09-03 22:02:31,410 [WARNING] Failures: [{'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'Initiative Implementation: id value of incorrect type: a0V6F00001gqMavUAE+', 'fields': ['Initiative_Implementa

## Phase 2 - CommPay_Supplier__c

In [11]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_CommPay_Supplier-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= CommPay_Supplier_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total CommPay_Supplier__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:03:27,775 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:03:27,777 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:03:27,779 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:03:27,780 INFO sqlalchemy.engine.Engine [generated in 0.00106s] ()
2026-09-03 22:03:27,783 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:03:27,784 INFO sqlalchemy.engine.Engine [generated in 0.00089s] ()
2026-09-03 22:03:27,785 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:03:27,786 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ()
2026-09-03 22:03:27,788 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:03:27,788 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_CommPay_Supplier-ALL]
;
2026-09-03 22:03:27,788 INFO sqlalchemy.en

2026-09-03 22:03:27,876 [INFO] Total CommPay_Supplier__c Records: 22
2026-09-03 22:03:27,878 [INFO] Preparing to upsert 22 records to CommPay_Supplier__c with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'CommPay_Supplier_Name__c', 'Licensee__c', 'Third_Party_Provider__c']
2026-09-03 22:03:28,725 [INFO] Upsert complete. Time taken: 0.85s
2026-09-03 22:03:28,726 [INFO] Success: 22, Failures: 0
2026-09-03 22:03:28,733 [INFO] Load Results:     success  created                  id errors
0      True     True  a11Bm000008sUR7IAM     []
1      True     True  a11Bm000008sUR8IAM     []
2      True     True  a11Bm000008sUR9IAM     []
3      True     True  a11Bm000008sURAIA2     []
4      True     True  a11Bm000008sURBIA2     []
5      True     True  a11Bm000008sURCIA2     []
6      True     True  a11Bm000008sURDIA2     []
7      True     True  a11Bm000008sUREIA2     []
8      True     True  a11Bm000008sURFIA2     []
9      True     True  a11Bm000008sU

## Phase 2 - Complained_Advisers__c

In [12]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Complained_Advisers-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= Complained_Advisers_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Complained_Advisers__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Complained_Advisers__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:04:02,715 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:04:02,716 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:04:02,716 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:04:02,716 INFO sqlalchemy.engine.Engine [generated in 0.00135s] ()
2026-09-03 22:04:02,716 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:04:02,724 INFO sqlalchemy.engine.Engine [generated in 0.00110s] ()
2026-09-03 22:04:02,726 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:04:02,727 INFO sqlalchemy.engine.Engine [generated in 0.00091s] ()
2026-09-03 22:04:02,730 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:04:02,731 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Complained_Advisers-ALL]
;
2026-09-03 22:04:02,731 INFO sqlalchemy

2026-09-03 22:04:03,367 [INFO] Total Complained_Advisers__c Records: 7724
2026-09-03 22:04:03,373 [INFO] Unique in Metazoa_External_ID__c : 7724
2026-09-03 22:04:03,488 [INFO] Preparing to upsert 7724 records to Complained_Advisers__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'EDP_External_Id__c', 'Case__c', 'Adviser__c']
2026-09-03 22:05:41,000 [INFO] Upsert complete. Time taken: 97.51s
2026-09-03 22:05:41,002 [INFO] Success: 7724, Failures: 0
2026-09-03 22:05:41,014 [INFO] Load Results:       success  created                  id errors
0        True     True  a12Bm000009296rIAA     []
1        True     True  a12Bm000009296sIAA     []
2        True     True  a12Bm000009296tIAA     []
3        True     True  a12Bm000009296uIAA     []
4        True     True  a12Bm000009296vIAA     []
...       ...      ...                 ...    ...
7719     True     True  a12Bm0000092B7MIAU     []
7720     True     True  a12Bm0000092B7N

## Phase 2 - EnhancedLetterhead

In [13]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_EnhancedLetterhead-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= ENHANCEDLETTERHEAD_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total EnhancedLetterhead Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='EnhancedLetterhead', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:06:18,316 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:06:18,318 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:06:18,320 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:06:18,321 INFO sqlalchemy.engine.Engine [generated in 0.00098s] ()
2026-09-03 22:06:18,324 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:06:18,325 INFO sqlalchemy.engine.Engine [generated in 0.00086s] ()


2026-09-03 22:06:18,327 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:06:18,328 INFO sqlalchemy.engine.Engine [generated in 0.00108s] ()
2026-09-03 22:06:18,328 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:06:18,328 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_EnhancedLetterhead-ALL]
;
2026-09-03 22:06:18,328 INFO sqlalchemy.engine.Engine [generated in 0.00139s] ()
2026-09-03 22:06:18,360 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 22:06:18,365 [INFO] Total EnhancedLetterhead Records: 1
2026-09-03 22:06:18,368 [INFO] Preparing to upsert 1 records to EnhancedLetterhead with fields ['CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Description', 'LetterheadFooter', 'LetterheadHeader', 'Name']
2026-09-03 22:06:25,206 [INFO] Upsert complete. Time taken: 6.84s
2026-09-03 22:06:25,207 [INFO] Success: 1, Failures: 0
2026-09-03 22:06:25,210 [INFO] Load Results:    success  created                  id errors
0     True     True  0RnBm0000001nbNKAQ     []


## Phase 2 - EmailTemplate

In [14]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_EmailTemplate-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= EMAILTEMPLATE_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total EmailTemplate Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='CommPay_Supplier__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='EmailTemplate', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:06:54,551 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:06:54,552 INFO sqlalchemy.engine.Engine [raw sql] ()


2026-09-03 22:06:54,555 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:06:54,556 INFO sqlalchemy.engine.Engine [generated in 0.00124s] ()
2026-09-03 22:06:54,559 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:06:54,559 INFO sqlalchemy.engine.Engine [generated in 0.00064s] ()
2026-09-03 22:06:54,561 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:06:54,562 INFO sqlalchemy.engine.Engine [generated in 0.00085s] ()
2026-09-03 22:06:54,564 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:06:54,565 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_EmailTemplate-ALL]
;
2026-09-03 22:06:54,664 INFO sqlalchemy.engine.Engine [generated in 0.10009s] ()
2026-09-03 22:06:54,880 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 22:06:54,883 [INFO] Total EmailTemplate Records: 0
2026-09-03 22:06:54,884 [INFO] Preparing to upsert 0 records to EmailTemplate with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'ApiVersion', 'Body', 'Description', 'DeveloperName', 'Encoding', 'HtmlValue', 'IsActive', 'Markup', 'Name', 'RelatedEntityType', 'Subject', 'TemplateStyle', 'TemplateType', 'UiType', 'BrandTemplateId', 'EnhancedLetterheadId', 'FolderId']
2026-09-03 22:06:54,888 [ERROR] Error during upsert: data should not be empty for insert


ValueError: data should not be empty for insert

## Phase 2 - Event

In [15]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Event-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= Event_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Event Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Event', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Event', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:07:24,401 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:07:24,402 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:07:24,404 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:07:24,405 INFO sqlalchemy.engine.Engine [generated in 0.00110s] ()
2026-09-03 22:07:24,408 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:07:24,408 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()
2026-09-03 22:07:24,411 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:07:24,413 INFO sqlalchemy.engine.Engine [generated in 0.00079s] ()
2026-09-03 22:07:24,414 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:07:24,416 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Event-ALL]
;
2026-09-03 22:07:24,417 INFO sqlalchemy.engine.Engine

2026-09-03 22:07:25,458 [INFO] Total Event Records: 30
2026-09-03 22:07:25,458 [INFO] Unique in Metazoa_External_ID__c : 30
2026-09-03 22:07:25,472 [INFO] Preparing to upsert 30 records to Event with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'ActivityDate', 'ActivityDateTime', 'AdviceAssuranceTeamComments__c', 'ASN_Decision__c', 'Changed_By__c', 'Comments__c', 'Communication_Channel__c', 'Completed_Date__c', 'Contact_Date__c', 'Continue_to_Next_Phase__c', 'Created_from_Std_QA_batch__c', 'Date_registered_booked_sent__c', 'Description', 'DurationInMinutes', 'EDP_External_Id__c', 'EndDateTime', 'EventSubtype', 'Future_time_period_Month_Year__c', 'IsAllDayEvent', 'IsPrivate', 'IsRecurrence', 'IsRecurrence2Exception', 'IsReminderSet', 'IsVisibleInSelfService', 'Legal_Privileged_Confidential__c', 'Licensee_Supervisor_Comments__c', 'Location', 'Preferred_Name__c', 'Reason_Why__c', 'Received_completed_attended_Date__c', '

## Phase 2 - EventRelation

In [16]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_EventRelation-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= EventRelation_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total EventRelation Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='EventRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Event', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Question_Responses__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-08-31 17:46:30,821 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-08-31 17:46:30,823 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-08-31 17:46:30,823 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-08-31 17:46:30,823 INFO sqlalchemy.engine.Engine [generated in 0.00101s] ()
2026-08-31 17:46:30,833 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-08-31 17:46:30,834 INFO sqlalchemy.engine.Engine [generated in 0.00106s] ()
2026-08-31 17:46:30,836 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-08-31 17:46:30,837 INFO sqlalchemy.engine.Engine [generated in 0.00097s] ()
2026-08-31 17:46:30,871 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-08-31 17:46:30,872 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_EventRelation-ALL]
;
2026-08-31 17:46:30,873 INFO sqlalchemy.engin

2026-08-31 17:46:31,019 [INFO] Total EventRelation Records: 0
2026-08-31 17:46:31,019 [INFO] Preparing to upsert 0 records to EventRelation with fields ['IsInvitee', 'IsParent', 'IsWhat', 'RespondedDate', 'Response', 'Status', 'RelationId', 'EventId']
2026-08-31 17:46:31,019 [ERROR] Error during upsert: data should not be empty for insert


ValueError: data should not be empty for insert

## Phase 2 - NAPW__c

In [16]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_NAPW-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= NAPW_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total NAPW__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:08:21,202 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:08:21,203 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:08:21,205 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:08:21,206 INFO sqlalchemy.engine.Engine [generated in 0.00101s] ()
2026-09-03 22:08:21,210 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:08:21,211 INFO sqlalchemy.engine.Engine [generated in 0.00117s] ()
2026-09-03 22:08:21,213 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:08:21,214 INFO sqlalchemy.engine.Engine [generated in 0.00114s] ()
2026-09-03 22:08:21,216 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:08:21,217 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_NAPW-ALL]
;
2026-09-03 22:08:21,218 INFO sqlalchemy.engine.Engine 

2026-09-03 22:08:21,306 [INFO] Total NAPW__c Records: 219
2026-09-03 22:08:21,307 [INFO] Unique in Metazoa_External_ID__c : 219
2026-09-03 22:08:21,319 [INFO] Preparing to upsert 219 records to NAPW__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'advice_for__c', 'adviser_id__c', 'Advisername__c', 'AdviserSFID__c', 'area_of_advice__c', 'ASIC_AR_Number__c', 'best_interest__c', 'client_adviser__c', 'client_entity_name__c', 'client_id__c', 'distribution_channel__c', 'expiry_date__c', 'Firstname__c', 'fum__c', 'individual_or_group_cover__c', 'inside_or_outside_super__c', 'Lastname__c', 'life_sum_insured__c', 'napw_apir_asx_code__c', 'napw_external_research__c', 'new_or_existing__c', 'outcome__c', 'outcome_commentary__c', 'outcome_date__c', 'partner_entity_name__c', 'partner_id__c', 'primary_group__c', 'product_name__c', 'salary_continuance_sum_insured__c', 'submission_date__c', 'tmd_link__c', 'tpd_sum_insured__c', '

## Phase 2 - Qualification_Type__c

In [17]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Qualification_Type-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= Qualification_TYPE_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Qualification_Type__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Qualification_Type__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Qualification_Type__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:09:10,218 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:09:10,219 INFO sqlalchemy.engine.Engine [raw sql] ()


2026-09-03 22:09:10,222 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:09:10,223 INFO sqlalchemy.engine.Engine [generated in 0.00108s] ()
2026-09-03 22:09:10,225 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:09:10,227 INFO sqlalchemy.engine.Engine [generated in 0.00118s] ()
2026-09-03 22:09:10,229 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:09:10,230 INFO sqlalchemy.engine.Engine [generated in 0.00097s] ()
2026-09-03 22:09:10,232 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:09:10,234 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Qualification_Type-ALL]
;
2026-09-03 22:09:10,235 INFO sqlalchemy.engine.Engine [generated in 0.00106s] ()
2026-09-03 22:09:10,298 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 22:09:10,302 [INFO] Total Qualification_Type__c Records: 95
2026-09-03 22:09:10,306 [INFO] Preparing to upsert 95 records to Qualification_Type__c with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'EDP_External_Id__c', 'Name']
2026-09-03 22:09:11,212 [INFO] Upsert complete. Time taken: 0.91s
2026-09-03 22:09:11,215 [INFO] Success: 95, Failures: 0
2026-09-03 22:09:11,220 [INFO] Load Results:     success  created                  id errors
0      True     True  a1UBm000002KUIXMA4     []
1      True     True  a1UBm000002KUIYMA4     []
2      True     True  a1UBm000002KUIZMA4     []
3      True     True  a1UBm000002KUIaMAO     []
4      True     True  a1UBm000002KUIbMAO     []
..      ...      ...                 ...    ...
90     True     True  a1UBm000002KUJzMAO     []
91     True     True  a1UBm000002KUK0MAO     []
92     True     True  a1UBm000002KUK1MAO     []
93     True     True  a1UBm000002KUK2MAO     []
94     True     True  a

## Phase 2 - Qualification__c

In [18]:
# Assurance_Review__c object
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Qualification-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= Qualification_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Qualification__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Qualification__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:09:46,194 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:09:46,195 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:09:46,197 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:09:46,198 INFO sqlalchemy.engine.Engine [generated in 0.00094s] ()
2026-09-03 22:09:46,201 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:09:46,202 INFO sqlalchemy.engine.Engine [generated in 0.00083s] ()
2026-09-03 22:09:46,204 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:09:46,204 INFO sqlalchemy.engine.Engine [generated in 0.00084s] ()
2026-09-03 22:09:46,208 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:09:46,208 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Qualification-ALL]
;
2026-09-03 22:09:46,208 INFO sqlalchemy.engin

2026-09-03 22:09:46,553 [INFO] Total Qualification__c Records: 3336
2026-09-03 22:09:46,556 [INFO] Unique in Metazoa_External_ID__c : 3336
2026-09-03 22:09:46,611 [INFO] Preparing to upsert 3336 records to Qualification__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'EDP_External_Id__c', 'Other_Qualification__c', 'Other_Registered_Training_Organisation__c', 'Qualification_Course_Status__c', 'Qualification_TypeNew__c', 'Specialist_Accreditation__c', 'Subject_Name__c', 'TASA__c', 'Year_Obtained__c', 'Adviser__c', 'Qualification_Type__c', 'Registered_Training_Organisation__c']
2026-09-03 22:09:57,897 [INFO] Upsert complete. Time taken: 11.28s
2026-09-03 22:09:57,899 [INFO] Success: 3336, Failures: 0
2026-09-03 22:09:57,908 [INFO] Load Results:       success  created                  id errors
0        True     True  a1VBm00000BeFhxMAF     []
1        True     True  a1VBm00000BeFhyMAF     []
2        True     True  a1VBm00000

## 70 After Assurance_Review__c - ASN_Indicator_History__c

In [19]:
"""
reference_to
["Assurance_Review__c"]
["Contact"]
["Group", "User"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_ASN_Indicator_History-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= ASN_Indicator_History_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total ASN_Indicator_History__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='TaskRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='ASN_Indicator_History__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:10:36,694 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:10:36,695 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:10:36,697 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:10:36,699 INFO sqlalchemy.engine.Engine [generated in 0.00148s] ()
2026-09-03 22:10:36,701 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:10:36,702 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ()
2026-09-03 22:10:36,704 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:10:36,705 INFO sqlalchemy.engine.Engine [generated in 0.00088s] ()
2026-09-03 22:10:36,707 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:10:36,707 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_ASN_Indicator_History-ALL]
;
2026-09-03 22:10:36,707 INFO sqlalche

2026-09-03 22:10:37,002 [INFO] Total ASN_Indicator_History__c Records: 1607
2026-09-03 22:10:37,003 [INFO] Unique in Metazoa_External_ID__c : 1607
2026-09-03 22:10:37,022 [INFO] Preparing to upsert 1607 records to ASN_Indicator_History__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'ASN__c', 'Review_Count_or_Average__c', 'Type__c', 'Assurance_Review__c', 'Adviser__c']
2026-09-03 22:11:03,437 [INFO] Upsert complete. Time taken: 26.41s
2026-09-03 22:11:03,438 [INFO] Success: 1607, Failures: 0
2026-09-03 22:11:03,446 [INFO] Load Results:       success  created                  id errors
0        True     True  a0YBm000008Z1XhMAK     []
1        True     True  a0YBm000008Z1XiMAK     []
2        True     True  a0YBm000008Z1XjMAK     []
3        True     True  a0YBm000008Z1XkMAK     []
4        True     True  a0YBm000008Z1XlMAK     []
...       ...      ...                 ...    ...
1602     True     True  a0YBm0000

## 72 After Assurance_Review__c - Reviews_Accounted__c

In [20]:
"""
reference_to
["ASN_Indicator_History__c"]
["Assurance_Review__c"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Reviews_Accounted-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= REVIEWS_ACCOUNTED_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Reviews_Accounted__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='TaskRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Reviews_Accounted__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:11:41,005 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:11:41,009 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:11:41,011 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:11:41,012 INFO sqlalchemy.engine.Engine [generated in 0.00104s] ()


2026-09-03 22:11:41,015 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:11:41,016 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()
2026-09-03 22:11:41,018 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:11:41,019 INFO sqlalchemy.engine.Engine [generated in 0.00094s] ()
2026-09-03 22:11:41,021 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:11:41,023 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Reviews_Accounted-ALL]
;
2026-09-03 22:11:41,024 INFO sqlalchemy.engine.Engine [generated in 0.00138s] ()
2026-09-03 22:11:41,412 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 22:11:41,419 [INFO] Total Reviews_Accounted__c Records: 5077
2026-09-03 22:11:41,421 [INFO] Unique in Metazoa_External_ID__c : 5077
2026-09-03 22:11:41,459 [INFO] Preparing to upsert 5077 records to Reviews_Accounted__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Indicator_Score__c', 'ASN_Indicator_History__c', 'Assurance_Review__c']
2026-09-03 22:11:47,862 [INFO] Upsert complete. Time taken: 6.40s
2026-09-03 22:11:47,864 [INFO] Success: 4810, Failures: 267
2026-09-03 22:11:47,866 [WARNING] Failures: [{'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'ASN Indicator History: id value of incorrect type: a1eMq000001OvpOIAS+', 'fields': ['ASN_Indicator_History__c']}]}, {'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'ASN Indicator History: id value of incorrect type: a1eMq000001OvpOIAS+', 'fields': ['ASN_Indic

## 80 After Question_Responses__c - Snapshot_Data__c

In [21]:
"""
reference_to
["Area_of_Advice__c"]
["Case"]
["Group", "User"]
["Question_Responses__c"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Snapshot_Data-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= Snapshot_Data_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Snapshot_Data__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Snapshot_Data__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:12:54,744 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:12:54,745 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:12:54,746 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:12:54,747 INFO sqlalchemy.engine.Engine [generated in 0.00078s] ()
2026-09-03 22:12:54,750 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:12:54,751 INFO sqlalchemy.engine.Engine [generated in 0.00096s] ()
2026-09-03 22:12:54,753 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:12:54,754 INFO sqlalchemy.engine.Engine [generated in 0.00105s] ()
2026-09-03 22:12:54,756 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:12:54,758 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Snapshot_Data-ALL]
;
2026-09-03 22:12:54,758 INFO sqlalchemy.engin

2026-09-03 22:23:58,323 [INFO] Total Snapshot_Data__c Records: 12080
2026-09-03 22:23:58,328 [INFO] Unique in Metazoa_External_ID__c : 12080
2026-09-03 22:23:58,627 [INFO] Preparing to upsert 12080 records to Snapshot_Data__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Case_Status__c', 'Case_Type__c', 'EDP_External_Id__c', 'Shapshot_Frequency__c', 'Snapshot_Date__c', 'Snapshot_Type__c', 'Status__c', 'Vetting_Reason__c', 'Vetting_Status__c', 'Area_of_Advice__c', 'Case__c', 'Incident__c']
2026-09-03 22:24:05,808 [INFO] Upsert complete. Time taken: 7.18s
2026-09-03 22:24:05,810 [INFO] Success: 957, Failures: 11123
2026-09-03 22:24:05,842 [WARNING] Failures: [{'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'Incident: id value of incorrect type: a0N6F00000X2m96UAB+', 'fields': ['Incident__c']}]}, {'success': False, 'created': False, 'id': None, 'errors': [{'stat

## 80 After Question_Responses__c - Response_Finding__c

In [22]:
"""
reference_to
["Master_Questions__c"]
["MasterQuestions_Targetedreviews__c"]
["Question_Responses__c"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
Event
EventRelation
Task
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Response_Finding-ALL]
;"""

from sf_target_fields import *
ls_tgt_objFields= Response_Finding_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Response_Finding__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='NAPW__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Response_Finding__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:25:53,487 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:25:53,489 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:25:53,492 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:25:53,493 INFO sqlalchemy.engine.Engine [generated in 0.00164s] ()
2026-09-03 22:25:53,496 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:25:53,497 INFO sqlalchemy.engine.Engine [generated in 0.00085s] ()
2026-09-03 22:25:53,499 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:25:53,500 INFO sqlalchemy.engine.Engine [generated in 0.00088s] ()
2026-09-03 22:25:53,502 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:25:53,503 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Response_Finding-ALL]
;
2026-09-03 22:25:53,504 INFO sqlalchemy.en

2026-09-03 22:25:54,087 [INFO] Total Response_Finding__c Records: 4
2026-09-03 22:25:54,088 [INFO] Unique in Metazoa_External_ID__c : 4
2026-09-03 22:25:54,088 [INFO] Preparing to upsert 4 records to Response_Finding__c with fields ['Metazoa_External_ID__c', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Actions_Taken__c', 'ASN_Indicator__c', 'Due_Date__c', 'Findings__c', 'Root_Cause__c', 'Master_Question__c', 'MasterQuestions_Targetedreviews__c', 'Incident__c']
2026-09-03 22:25:55,111 [INFO] Upsert complete. Time taken: 1.02s
2026-09-03 22:25:55,111 [INFO] Success: 0, Failures: 4
2026-09-03 22:25:55,111 [WARNING] Failures: [{'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'Question Response: id value of incorrect type: a0NMq000000XdLzMAK+', 'fields': ['Incident__c']}]}, {'success': False, 'created': False, 'id': None, 'errors': [{'statusCode': 'MALFORMED_ID', 'message': 'Question Response: id value of incorre

## 85 Before TASK - Vetting_Result__c

In [23]:
"""
reference_to
["Account"]
["Contact"]
["Group", "User"]
["RecordType"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Vetting_Result-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= Vetting_Result_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Vetting_Result__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='TaskRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Vetting_Result__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:27:18,865 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:27:18,866 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:27:18,868 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:27:18,869 INFO sqlalchemy.engine.Engine [generated in 0.00082s] ()


2026-09-03 22:27:18,873 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:27:18,873 INFO sqlalchemy.engine.Engine [generated in 0.00133s] ()
2026-09-03 22:27:18,873 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:27:18,873 INFO sqlalchemy.engine.Engine [generated in 0.00072s] ()
2026-09-03 22:27:18,873 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:27:18,873 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Vetting_Result-ALL]
;
2026-09-03 22:27:18,873 INFO sqlalchemy.engine.Engine [generated in 0.00087s] ()
2026-09-03 22:27:19,142 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 22:27:19,147 [INFO] Total Vetting_Result__c Records: 1428
2026-09-03 22:27:19,148 [INFO] Unique in Metazoa_External_ID__c : 1428
2026-09-03 22:27:19,175 [INFO] Preparing to upsert 1428 records to Vetting_Result__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'ARResubmitted__c', 'EDP_External_Id__c', 'File_Number__c', 'MandatoryQuestionsFail__c', 'Number_of_High_Issues__c', 'Number_of_Low_Issues__c', 'Number_of_Medium_issues__c', 'Phase_Number__c', 'Reason_for_Vetting__c', 'Total_Number_of_Issues__c', 'Vetting_Status_at_Finish__c', 'Vetting_Status_at_Start__c', 'Area_of_Advice__c', 'Assurance_Review__c']
2026-09-03 22:27:35,399 [INFO] Upsert complete. Time taken: 16.22s
2026-09-03 22:27:35,400 [INFO] Success: 1428, Failures: 0
2026-09-03 22:27:35,408 [INFO] Load Results:       success  created                  id errors
0        True     True  a1qBm000009DBTVIA4     []
1        True     True  a1qBm0000

## 85 Before TASK - Wealth_Central_Adoption__c

In [24]:
"""
reference_to
["Account"]
["Contact"]
["Group", "User"]
["RecordType"]
["User"]
"""
"""
object_api_name
EmailMessage
Event
EventRelation
Task
TaskRelation
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Wealth_Central_Adoption-ALL]
;"""

ls_tgt_objFields= Wealth_Central_Adoption_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Wealth_Central_Adoption__c Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='TaskRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Wealth_Central_Adoption__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:28:03,801 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:28:03,802 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:28:03,804 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:28:03,805 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()
2026-09-03 22:28:03,808 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:28:03,809 INFO sqlalchemy.engine.Engine [generated in 0.00078s] ()
2026-09-03 22:28:03,811 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:28:03,812 INFO sqlalchemy.engine.Engine [generated in 0.00084s] ()
2026-09-03 22:28:03,814 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:28:03,815 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Wealth_Central_Adoption-ALL]
;
2026-09-03 22:28:03,816 INFO sqlalc

2026-09-03 22:28:04,016 [INFO] Total Wealth_Central_Adoption__c Records: 807
2026-09-03 22:28:04,018 [INFO] Unique in Metazoa_External_ID__c : 807
2026-09-03 22:28:04,041 [INFO] Preparing to upsert 807 records to Wealth_Central_Adoption__c with fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Access_before_1_Oct_2021__c', 'Access_before_1_Oct_2021_C__c', 'Adviser_Moved_Practice__c', 'Advocate__c', 'Comments__c', 'Comments_Non_HTML__c', 'Committing_to_Adoption_Program__c', 'Good_candidate_for_WC__c', 'Historic_IOOF_pilot_participant__c', 'Level_of_WC_access_unlocked__c', 'MLC_Pilot_participant__c', 'Not_interested_ready__c', 'Peer_Group__c', 'Reason_Why__c', 'Successfully_embedded_WC__c', 'Terminated_access__c', 'Practice__c', 'Adviser__c', 'NominatedChampionAdviser__c', 'RecordTypeId', 'User__c']
2026-09-03 22:28:10,150 [INFO] Upsert complete. Time taken: 6.11s
2026-09-03 22:28:10,151 [INFO] Success: 799, Failures: 8
20

## 99 END - Adviser_ASN_Snapshot__c

In [25]:
"""
reference_to
["Contact"]
["Group", "User"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Adviser_ASN_Snapshot-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= Adviser_ASN_Snapshot_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Adviser_ASN_Snapshot__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Adviser_ASN_Snapshot__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Adviser_ASN_Snapshot__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:29:11,001 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)


2026-09-03 22:29:11,003 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:29:11,005 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:29:11,008 INFO sqlalchemy.engine.Engine [generated in 0.00192s] ()
2026-09-03 22:29:11,009 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:29:11,009 INFO sqlalchemy.engine.Engine [generated in 0.00094s] ()
2026-09-03 22:29:11,009 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:29:11,009 INFO sqlalchemy.engine.Engine [generated in 0.00081s] ()
2026-09-03 22:29:11,009 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:29:11,009 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Adviser_ASN_Snapshot-ALL]
;
2026-09-03 22:29:11,009 INFO sqlalchemy.engine.Engine [generated in 0.00090s] ()
2026-09-03 22:29:11,299 INFO sqlalchemy.engine.Engine COMMIT


2026-09-03 22:29:11,304 [INFO] Total Adviser_ASN_Snapshot__c Records: 4085
2026-09-03 22:29:11,344 [INFO] Preparing to upsert 4085 records to Adviser_ASN_Snapshot__c with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'ASN_Source__c', 'ASN_Start_Date__c', 'Execution_Time__c', 'Adviser__c']
2026-09-03 22:29:17,553 [INFO] Upsert complete. Time taken: 6.21s
2026-09-03 22:29:17,555 [INFO] Success: 4085, Failures: 0
2026-09-03 22:29:17,564 [INFO] Load Results:       success  created                  id errors
0        True     True  a2MBm000006pL3lMAE     []
1        True     True  a2MBm000006pL3mMAE     []
2        True     True  a2MBm000006pL3nMAE     []
3        True     True  a2MBm000006pL3oMAE     []
4        True     True  a2MBm000006pL3pMAE     []
...       ...      ...                 ...    ...
4080     True     True  a2MBm000006pM7ZMAU     []
4081     True     True  a2MBm000006pM7aMAE     []
4082     True     True  a2MBm000006pM7bMAE     [

## 99 END - Refund_Request__c

In [26]:
"""
reference_to
["Case"]
["Contact"]
["Group", "User"]
["Incident_File_Review__c"]
["RecordType"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Refund_Request-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= Refund_Request_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

logger.info(f"Total Refund_Request__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Refund_Request__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Refund_Request__c', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=10000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:29:47,648 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:29:47,649 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:29:47,651 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:29:47,652 INFO sqlalchemy.engine.Engine [generated in 0.00109s] ()
2026-09-03 22:29:47,655 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:29:47,655 INFO sqlalchemy.engine.Engine [generated in 0.00110s] ()
2026-09-03 22:29:47,658 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:29:47,659 INFO sqlalchemy.engine.Engine [generated in 0.00082s] ()
2026-09-03 22:29:47,661 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:29:47,662 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Refund_Request-ALL]
;
2026-09-03 22:29:47,663 INFO sqlalchemy.engi

2026-09-03 22:29:48,110 [INFO] Total Refund_Request__c Records: 919
2026-09-03 22:29:48,180 [INFO] Preparing to upsert 919 records to Refund_Request__c with fields ['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Alternative_payment_Confirmation_Date__c', 'Alternative_payment_Request_Sent_Date__c', 'At_Fault__c', 'Calculation_required__c', 'Client_Category__c', 'Comments__c', 'Confirm_payment_with_Trustee__c', 'Counterfactual__c', 'Date_Refund_Assessed__c', 'Date_Refund_Processed__c', 'Date_Refund_Submitted__c', 'Final_Refund_Amount__c', 'Form_Submitted_Date__c', 'Further_Information_provided__c', 'Further_information_received__c', 'Further_Information_required__c', 'Have_you_provided_bank_details__c', 'Interest__c', 'LICENSEE_NAME__c', 'Name_of_Client_s__c', 'Payment_confirmation_received__c', 'Payment_processing_method__c', 'Payment_request_lodged__c', 'Policy_Number__c', 'Policy_Status__c', 'Product_Platform_s_For_Refund_Payment__c', 'Product_Provid

## 99 END - Calculation__c

In [27]:
"""
reference_to
["RecordType"]
["Refund_Request__c"]
["User"]
"""
# object transfer from source to target.
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Calculation-ALL]
;"""

from sf_target_fields import *

ls_tgt_objFields= CACLCULATION_TARGET_FIELDS
# ls_tgt_objFields = [x for x in ls_tgt_objFields if x not in ['OwnerId', 'IsDeleted', 'SystemModstamp']]

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

2026-09-03 22:30:41,926 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-03 22:30:41,926 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-03 22:30:41,926 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-03 22:30:41,926 INFO sqlalchemy.engine.Engine [generated in 0.00071s] ()
2026-09-03 22:30:41,926 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-03 22:30:41,932 INFO sqlalchemy.engine.Engine [generated in 0.00095s] ()
2026-09-03 22:30:41,934 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-03 22:30:41,934 INFO sqlalchemy.engine.Engine [generated in 0.00073s] ()
2026-09-03 22:30:41,936 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-03 22:30:41,937 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Calculation-ALL]
;
2026-09-03 22:30:41,938 INFO sqlalchemy.engine.

,CreatedById,CreatedDate,LastModifiedById,LastModifiedDate,Actual_Reference_portfolio__c,Additional_Comments__c,Basic__c,Cancellation_reason__c,Client_Detriment__c,Complex__c,...,QA_required__c,Re_Calculation_required__c,Refund_In_Scope__c,Risk_profile_specify_G_D_split__c,Stage__c,Supporting_comments__c,RecordTypeId,Refund_Reference__c,Assigned_To__c,QA_Assigned_To__c
0,005Bm00000Y8Ka4IAF,2025-09-29T02:21:18.000+0000,005W2000000K2sxIAC,2025-10-27T04:34:37.000+0000,None,None,false,None,false,false,...,None,false,39.86,None,Completed,None,012Bm0000091qVdIAI,a2YBm000001KU3BMAW,005W2000000K2jyIAC,None
1,005Bm00000Y8KZpIAN,2026-08-17T03:12:32.000+0000,005W2000000K2jmIAC,2026-08-21T00:19:20.000+0000,None,None,false,None,false,false,...,None,false,4342.85,None,Completed,None,012Bm0000091qVdIAI,a2YBm000001KU2BMAW,005W2000000K2jyIAC,None
2,005W2000000K2mBIAS,2026-06-19T09:49:37.000+0000,005W2000000K2jmIAC,2026-06-30T23:49:38.000+0000,None,None,false,None,false,false,...,None,false,865.79,None,Completed,None,,a2YBm000001KTtSMAW,005W2000000K2jyIAC,None
3,005W2000000K2gwIAC,2026-06-23T00:32:22.000+0000,005W2000000K2jmIAC,2026-07-22T05:58:17.000+0000,None,None,false,None,false,false,...,None,false,0.0,None,Completed,None,012Bm0000091qVdIAI,a2YBm000001KTtWMAW,005W2000000K2jyIAC,None
4,005Bm00000Y8KCfIAN,2026-08-21T00:40:11.000+0000,005W2000000K2jmIAC,2026-08-26T04:57:33.000+0000,None,None,false,None,false,false,...,None,false,3078.53,None,Completed,Compensation Breakdown (also shown in the atta...,012Bm0000091qVdIAI,a2YBm000001KTzRMAW,005W2000000K2jyIAC,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
452,005Bm00000Y8J6kIAF,2026-09-01T05:18:18.000+0000,005Bm00000Y8J6kIAF,2026-09-01T05:18:18.000+0000,None,None,false,None,false,false,...,None,false,None,None,Assessment,None,012Bm0000091qVdIAI,a2YBm000001KTwOMAW,005W2000000K2jyIAC,None
453,005Bm00000Y8J6kIAF,2026-09-01T05:18:18.000+0000,005Bm00000Y8J6kIAF,2026-09-01T05:18:18.000+0000,None,None,false,None,false,false,...,None,false,None,None,Assessment,None,012Bm0000091qVdIAI,a2YBm000001KTwNMAW,005W2000000K2jyIAC,None
454,005Bm00000Y8J6kIAF,2026-09-01T06:57:50.000+0000,005Bm00000Y8J6kIAF,2026-09-01T06:57:50.000+0000,None,None,false,None,false,false,...,None,false,None,None,Assessment,None,012Bm0000091qVdIAI,a2YBm000001KTwTMAW,005W2000000K2jyIAC,None
455,005Bm00000Y8J6kIAF,2026-09-02T01:08:46.000+0000,005Bm00000Y8J6kIAF,2026-09-02T01:08:46.000+0000,None,None,false,None,false,false,...,None,false,None,None,Assessment,None,012Bm0000091qVdIAI,a2YBm000001KTwPMAW,005W2000000K2jyIAC,None


In [28]:

logger.info(f"Total Calculation__c Records: {df_transformed.shape[0]}")
# logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Calculation__c', reference_field='Id', fields=ls_tgt_objFields, batch_size=10000, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-03 22:30:52,805 [INFO] Total Calculation__c Records: 457
2026-09-03 22:30:52,828 [INFO] Preparing to upsert 457 records to Calculation__c with fields ['CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'Actual_Reference_portfolio__c', 'Additional_Comments__c', 'Basic__c', 'Cancellation_reason__c', 'Client_Detriment__c', 'Complex__c', 'Counterfactual__c', 'Final_Refund_Amount__c', 'Interest__c', 'Interest_earnings_end_date__c', 'Location_supporting_documents__c', 'Product__c', 'QA_Completed_Date__c', 'QA_Output__c', 'QA_required__c', 'Re_Calculation_required__c', 'Refund_In_Scope__c', 'Risk_profile_specify_G_D_split__c', 'Stage__c', 'Supporting_comments__c', 'RecordTypeId', 'Refund_Reference__c', 'Assigned_To__c', 'QA_Assigned_To__c']
2026-09-03 22:30:58,960 [INFO] Upsert complete. Time taken: 6.13s
2026-09-03 22:30:58,961 [INFO] Success: 447, Failures: 10
2026-09-03 22:30:58,962 [WARNING] Failures: [{'success': False, 'created': False, 'id': None, 'errors': [

## 90 END - Task - RUN 1 - INSERT

In [7]:
"""
reference_to
["A1_Support_Staff_Registrations__c", "ANZ_Lookback__c", "ASF_CSA_Case__c", "ASIC_Breach__c", "Account", "Accreditations__c", "AdviserONE_XPLAN_Registrations__c", "Adviser_Academy__c", "Advisers_Approved_Non_standard_Templates__c", "Application_Comment__c", "Approved_Non_standard_Template__c", "Area_of_Advice__c", "Article_Auto_Archival__c", "Asset", "AssetRelationship", "Assurance_Review__c", "AttributeCategory", "AttributeCategoryAttribute", "AttributeDefinition", "AttributePicklist", "AttributePicklistValue", "Australian_Public_Holiday__c", "Breach_Reporting__c", "CSA_Fee_Switch_Off_History__c", "CSOAdviser__c", "CSO__c", "CalculationMatrix", "CalculationMatrixVersion", "CalculationProcedure", "CalculationProcedureVersion", "Campaign", "Case", "Client_Details__c", "Client_Service_Model_Implementation__c", "Complained_Advisers__c", "Consequence_Management_Forum_Entry__c", "ContactRequest", "Contract", "ContractLineItem", "Control_Review__c", "CreditMemo", "DelegatedAccount", "Email_Log__c", "EngagementAttendee", "EngagementInteraction", "EngagementTopic", "Entitlement", "ExpressionSet", "ExpressionSetVersion", "Financial_Crime_Team__c", "GoalAssignment", "GoalDefinition", "Image", "Incident_File_Review__c", "Invoice", "Item__c", "Library_Text__c", "ListEmail", "Location", "MasterQuestions_Targetedreviews__c", "Master_Item__c", "Master_Questions__c", "Master_Topic__c", "Master_workshop__c", "Module__c", "NAPW__c", "Opportunity", "Order", "PEP__c", "PSA__c", "Paraplanning__c", "Person__c", "ProcessException", "Product2", "ProductAttributeDefinition", "ProductCatalog", "ProductClassification", "ProductClassificationAttr", "ProductComponentGroup", "ProductSellingModel", "QA_Assignment_Hisotry__c", "QA__c", "QDC_Training__c", "Qualification_Type__c", "Qualification__c", "Question_Group__c", "Question_Responses__c", "Quote", "RecordAlert", "Referral_Partner__c", "Registered_Training_Organisation__c", "Response_Finding__c", "Reviewer_Accreditation__c", "SecondaryPracticeAdviserHistory__c", "ServiceContract", "Solution", "Special_Conditions__c", "Target__c", "Topic__c", "Tracking__c", "Universal_Audit_Log__c", "Vetting_Result__c", "Wealth_Central_Adoption__c", "Wealth_Central__c", "Workshop__c", "bar__Backup_Object__c", "bar__Backup_Policy__c", "bridges_rpt_referral2__c", "bridges_rpt_referral__c", "bridges_rpt_referral_revenue2__c", "bridges_rpt_referral_revenue__c", "bridges_rpt_refunds__c", "bridges_tpo_commpay__c", "bridges_tpo_entity__c", "bridges_tpo_extractdate__c", "bridges_tpo_master_member__c", "bridges_tpo_member_review__c", "bridges_tpo_member_securities__c", "bridges_tpo_thread_task__c", "bridges_tpo_thread_task_bfpe__c", "bridges_tpo_user__c", "hmq__Headlines__c", "metazoa3__snapshot_asset__c", "metazoa3__snapshot_push__c", "metazoa3__snapshot_sprint__c", "metazoa3__snapshot_story__c", "unleashformula__Formula_Mapping__c"]
["Account"]
["ASN_Indicator_History__c"]
["Assurance_Review__c"]
["Contact", "Lead"]
["Group", "User"]
["Question_Responses__c"]
["RecordType"]
["Response_Finding__c"]
["Task"]
["User"]
"""
"""
object_api_name
EmailMessage
Task
TaskRelation
"""

'\nobject_api_name\nEmailMessage\nTask\nTaskRelation\n'

In [19]:
# New Task without WhatId - 834 - 142m
# q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId] = ''
# ;"""
# New Task with WhatId - 001 - Account
# q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like '001%' and [WhatId] not like '%+'
# ;"""
# New Task with WhatId - 500 - Case
# q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like '500%' and [WhatId] not like '%+'
# ;"""
# New Task with WhatId - a09 - Assurance_Review__c
q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like 'a09%' and [WhatId] not like '%+'
;"""
# New Task with WhatId - a12 - ASF_CSA_Case__c
# q_sql_select_src_tbl = f"""select * from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like 'a12%' and [WhatId] not like '%+'
# ;"""

from sf_target_fields import *

ls_tgt_objFields= Task_TARGET_FIELDS

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

2026-09-04 02:07:02,847 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-04 02:07:02,847 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-04 02:07:02,847 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-04 02:07:02,847 INFO sqlalchemy.engine.Engine [generated in 0.00139s] ()
2026-09-04 02:07:02,847 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-04 02:07:02,857 INFO sqlalchemy.engine.Engine [generated in 0.00130s] ()
2026-09-04 02:07:02,859 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-04 02:07:02,860 INFO sqlalchemy.engine.Engine [generated in 0.00086s] ()
2026-09-04 02:07:02,863 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 02:07:02,864 INFO sqlalchemy.engine.Engine select * from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like 'a09%' and [WhatId

,Metazoa_External_ID__c,OwnerId,CreatedById,CreatedDate,LastModifiedById,LastModifiedDate,ActivityDate,AdviceAssuranceTeamComments__c,ASN_Decision__c,CallDisposition,...,Task_Owner__c,Technical_Proficiency_Rating__c,Type,WhatId,ASN_Indicator_History__c,Assurance_Review__c,Question_Response__c,RecordTypeId,Response_Finding__c,Colleague__c
0,00T6F00007VtbbIUAR,005W2000000K2eKIAS,005W2000000K2eKIAS,2021-08-31T07:27:54.000+0000,005W2000000K2eKIAS,2021-08-31T07:27:55.000+0000,2021-08-31,None,None,None,...,None,None,None,a0oBm000004xZAyIAM,None,None,None,012Bm0000092FyfIAE,None,None
1,00T6F00007ZLwARUA1,005W2000000K2pvIAC,005W2000000K2pvIAC,2021-10-25T01:57:06.000+0000,005W2000000K2pvIAC,2021-10-25T01:57:06.000+0000,2021-10-25,None,None,None,...,None,None,None,a0oBm000004xZBUIA2,None,None,None,012Bm0000092FyfIAE,None,None
2,00T6F00007fDlrFUAS,005W2000000K2hfIAC,005W2000000K2hfIAC,2022-02-13T22:46:59.000+0000,005W2000000K2hfIAC,2022-02-13T22:46:59.000+0000,2022-02-14,None,None,None,...,None,None,None,a0oBm000004xZBUIA2,None,None,None,012Bm0000092FyfIAE,None,None
3,00T6F00007X5XbWUAV,005W2000000K2pvIAC,005W2000000K2pvIAC,2021-09-10T00:18:25.000+0000,005W2000000K2pvIAC,2021-09-10T00:18:25.000+0000,2021-09-10,None,None,None,...,None,None,None,a0oBm000004xZBUIA2,None,None,None,012Bm0000092FyfIAE,None,None
4,00T6F00007YElsxUAD,005W2000000K2pmIAC,005W2000000K2pmIAC,2021-09-28T03:41:02.000+0000,005W2000000K2pmIAC,2021-09-28T03:41:02.000+0000,2021-09-28,None,None,None,...,None,None,None,a0oBm000004xYpnIAE,None,None,None,012Bm0000092FyfIAE,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22653,00TMq00000tqVbDMAU,005W2000000K2fNIAS,005W2000000K2fNIAS,2026-08-16T23:42:05.000+0000,005W2000000K2fNIAS,2026-08-24T02:35:18.000+0000,2026-08-31,None,None,None,...,None,None,None,a0oBm000004xZSyIAM,None,a0oBm000004xZSyIAM,a0NMq000005LhQdMAK+,012W2000000AEpmIAG,a1aMq00000YHiInIAL+,005Bm00000Y8KMtIAN
22654,00TMq00000u3A7YMAU,005W2000000K2j1IAC,005W2000000K2fNIAS,2026-08-20T00:29:51.000+0000,005W2000000K2fNIAS,2026-08-20T00:29:51.000+0000,None,None,None,None,...,None,None,None,a0oBm000004xZSyIAM,None,a0oBm000004xZSyIAM,None,012W2000000AEpjIAG,None,None
22655,00TMq00000u3A7ZMAU,005Bm00000Y8KMtIAN,005W2000000K2fNIAS,2026-08-20T00:29:51.000+0000,005W2000000K2fNIAS,2026-08-20T00:29:51.000+0000,None,None,None,None,...,None,None,None,a0oBm000004xZSyIAM,None,a0oBm000004xZSyIAM,None,012W2000000AEpjIAG,None,None
22656,00TMq00000uE37aMAC,005Bm00000Y8KMtIAN,005W2000000K2fNIAS,2026-08-24T02:34:59.000+0000,005W2000000K2fNIAS,2026-08-24T02:34:59.000+0000,None,None,None,None,...,None,None,None,a0oBm000004xZSyIAM,None,a0oBm000004xZSyIAM,None,012W2000000AEplIAG,None,None


In [ ]:

logger.info(f"Total Task Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='TaskRelation', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger)
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Task', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=2000, logger=logger)
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

## 90 END - Task - RUN 2 - UPSERT on WhatId
### Run only if ncessary

In [33]:
"""
Archived records (IsDeleted=false but are not retrievable by SOQL queries) are not returned by the Salesforce API. To retrieve archived records, you must use the Bulk API or the SOAP API.
When attempting to INSERT them with blank Id, the operation will fail with 'DUPLICATE_VALUE". Since the Metazoa_External_ID__c is unique, the upsert will fail with 'DUPLICATE_VALUE' error.
To update WhatId, need an UPSERT with Metazoa_External_ID__c.
"""
# q_sql_select_src_tbl = f"""select [Id],[WhatId] from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] > '' and [WhatId] > '' and [WhatId] not like '%+'
# ;"""
# Task with WhatId - 001 - Account - 1557
# q_sql_select_src_tbl = f"""select [Metazoa_External_ID__c],[WhatId] from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like '001%' and [WhatId] not like '%+'
# ;"""
# Task with WhatId - 500 - Case - 36
# q_sql_select_src_tbl = f"""select [Metazoa_External_ID__c],[WhatId] from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like '500%' and [WhatId] not like '%+'
# ;"""
# Task with WhatId - a09 - Assurance_Review__c - 20293 (230450 - 112m
# q_sql_select_src_tbl = f"""select [Metazoa_External_ID__c],[WhatId] from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like 'a09%' and [WhatId] not like '%+'
# ;"""
# Task with WhatId - a12 - ASF_CSA_Case__c - 19 (1149)   - 3m
# q_sql_select_src_tbl = f"""select [Metazoa_External_ID__c],[WhatId] from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like 'a12%' and [WhatId] not like '%+'
# ;"""
# Task with WhatId - a26 - ASF_CSA_Case_dv__c - 2 (232_ - 8s
# q_sql_select_src_tbl = f"""select [Metazoa_External_ID__c],[WhatId] from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like 'a26%' and [WhatId] not like '%+'
# ;"""
# Task with WhatId - a29 - ASF_CSA_Case_dv__c - 0 (2)
# q_sql_select_src_tbl = f"""select [Metazoa_External_ID__c],[WhatId] from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like 'a29%' and [WhatId] not like '%+'
# ;"""
# Task with WhatId - a2A - Refund_Request__c - 240 (671) - 9s
q_sql_select_src_tbl = f"""select [Metazoa_External_ID__c],[WhatId] from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [WhatId-src] like 'a2A%' and [WhatId] not like '%+'
;"""

ls_tgt_objFields= ['Metazoa_External_ID__c','WhatId']

engine = get_engine(connection_string)
with engine.begin() as conn:
    src_select_tbl = conn.execute(text(q_sql_select_src_tbl))
    # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
    # columns = src_select_tbl.keys()
    rows = src_select_tbl.fetchall()
    df_src_tbl = pd.DataFrame(rows, columns=src_select_tbl.keys())
    conn.commit()
    conn.close()

df_transformed = df_src_tbl[ls_tgt_objFields].copy()  # Start with only target fields
df_transformed = df_transformed.replace({np.nan: None})

df_transformed

2026-09-02 13:28:13,323 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-02 13:28:13,324 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-02 13:28:13,327 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-02 13:28:13,327 INFO sqlalchemy.engine.Engine [generated in 0.00092s] ()
2026-09-02 13:28:13,327 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-02 13:28:13,327 INFO sqlalchemy.engine.Engine [generated in 0.00148s] ()
2026-09-02 13:28:13,327 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-02 13:28:13,327 INFO sqlalchemy.engine.Engine [generated in 0.00064s] ()
2026-09-02 13:28:13,327 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-02 13:28:13,327 INFO sqlalchemy.engine.Engine select [Metazoa_External_ID__c],[WhatId] from [SalesforceMigration].[sfrhd].[vw_upsert_Task-ALL] where [Id] = '' and [Wha

,Metazoa_External_ID__c,WhatId
0,00TMq00000XRZwPMAX,a2YBm000001Jwk3MAC
1,00TMq00000XRX3MMAX,a2YBm000001JwhjMAC
2,00TMq00000XSGEwMAP,a2YBm000001JwgsMAC
3,00TMq00000Y0KBFMA3,a2YBm000001JwjaMAC
4,00TMq00000XRfIeMAL,a2YBm000001JwlWMAS
...,...,...
235,00TMq00000WpdUoMAJ,a2YBm000001JwnxMAC
236,00TMq00000XPo8IMAT,a2YBm000001Jwj7MAC
237,00TMq00000X9GZLMA3,a2YBm000001Jwj8MAC
238,00TMq00000XQzPyMAL,a2YBm000001JwjqMAC


In [45]:

logger.info(f"Total Task Records: {df_transformed.shape[0]}")
logger.info(f"Unique in Metazoa_External_ID__c : {df_transformed['Metazoa_External_ID__c'].nunique()}")

# Step 4: Load transformed data to destination org via upsert
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Task', reference_field='Id', fields=ls_tgt_objFields, batch_size=100, logger=logger,mode='UPSERT')
df_load_results = load_data(dst_sf_instance, df_transformed, object_name='Task', reference_field='Metazoa_External_ID__c', fields=ls_tgt_objFields, batch_size=2000, logger=logger,mode='UPSERT')
# df_load_results = load_data(dst_sf_instance, df_transformed, object_name='GroupMember', reference_field='UserOrGroupId', fields=ls_tgt_objFields, batch_size=10, logger=logger)

# Optional: Log load results
logger.info(f"Load Results: {df_load_results}")

2026-09-02 16:28:54,275 [INFO] Total Task Records: 834
2026-09-02 16:28:54,277 [INFO] Unique in Metazoa_External_ID__c : 834
2026-09-02 16:28:54,279 [ERROR] Error selecting fields ['Metazoa_External_ID__c', 'OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'ActivityDate', 'AdviceAssuranceTeamComments__c', 'ASN_Decision__c', 'CallDisposition', 'CallDurationInSeconds', 'CallObject', 'CallType', 'Changed_By__c', 'Comments__c', 'Communication_Channel__c', 'Completed_Date__c', 'Contact_Date__c', 'Continue_to_Next_Phase__c', 'Created_from_Std_QA_batch__c', 'Date_registered_booked_sent__c', 'Description', 'EDP_External_Id__c', 'Future_time_period_Month_Year__c', 'IsReminderSet', 'IsVisibleInSelfService', 'Legal_Privileged_Confidential__c', 'Licensee_Supervisor_Comments__c', 'Preferred_Name__c', 'Priority', 'Reason_Why__c', 'Received_completed_attended_Date__c', 'RecurrenceDayOfMonth', 'RecurrenceDayOfWeekMask', 'RecurrenceEndDateOnly', 'RecurrenceInstance', 'Rec

KeyError: "['OwnerId', 'CreatedById', 'CreatedDate', 'LastModifiedById', 'LastModifiedDate', 'ActivityDate', 'AdviceAssuranceTeamComments__c', 'ASN_Decision__c', 'CallDisposition', 'CallDurationInSeconds', 'CallObject', 'CallType', 'Changed_By__c', 'Comments__c', 'Communication_Channel__c', 'Completed_Date__c', 'Contact_Date__c', 'Continue_to_Next_Phase__c', 'Created_from_Std_QA_batch__c', 'Date_registered_booked_sent__c', 'Description', 'EDP_External_Id__c', 'Future_time_period_Month_Year__c', 'IsReminderSet', 'IsVisibleInSelfService', 'Legal_Privileged_Confidential__c', 'Licensee_Supervisor_Comments__c', 'Preferred_Name__c', 'Priority', 'Reason_Why__c', 'Received_completed_attended_Date__c', 'RecurrenceDayOfMonth', 'RecurrenceDayOfWeekMask', 'RecurrenceEndDateOnly', 'RecurrenceInstance', 'RecurrenceInterval', 'RecurrenceMonthOfYear', 'RecurrenceRegeneratedType', 'RecurrenceStartDateOnly', 'RecurrenceTimeZoneSidKey', 'RecurrenceType', 'ReminderDateTime', 'Resource_Hours__c', 'Section_